# Bowen RC beam impact — Stage 1: source-traceable reference setup

**Goal:** Build and validate the geometrical/material-point setup for Tu Bowen's thesis **Chapter 3, §3.1.1, Figures 3-1–3-2**. This notebook **does not claim to reproduce Figure 3-3 or Table 3-1 yet**. Contact mechanics, PDROD, and dynamic solver follow in subsequent stages.

**Citation:** Tu Bowen (2025), *Research on impact damage of concrete beams based on Peridynamics*, Southwest Jiaotong University, Chapter 3, pp. 30–32; theory Chapter 2.

**Traceability labels:** `THESIS` = explicitly documented; `DERIVED` = arithmetic consequence; `ASSUMED` = not specified sufficiently in source and must be checked; `TARGET` = published output to compare only after running an independently verified solver.

**Source cautions:** Thesis Figure 3-1 specifies Ø6×2 and a 80 mm internal vertical dimension but does not unambiguously dimension the two transverse bar positions or all support/contact implementation details. We do **not** claim bar positions or the illustrative initial impactor clearance are exact.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, asdict
import json

np.set_printoptions(precision=5, suppress=True)
OUT = Path('bowen_stage1_outputs'); OUT.mkdir(exist_ok=True)


## A. Chapter 3 reference parameters (SI units)

The model is **600 mm long × 70 mm wide × 100 mm deep**, discretized into **120 × 14 × 20 = 33,600** PD particles at the *cell centers*. The original coordinates themselves are not listed in the thesis; a regular cell-centered grid is a reproducible construction.

**Supports:** simple supports with 500 mm spacing, centered at x=50 and 550 mm (from Figure 3-1). Their contact width and constrained particle-layer selection are **not** specified in §3.1.1.

**Steel:** two Ø6 mm longitudinal rods; Figure 3-1 shows an 80 mm vertical dimension and bar lines near the bottom. We **provisionally** set the rod centerline at z=10 mm and transverse centers at y=20 and 50 mm. Review these against the original experiment or author's CAD/source.

**Impactor:** 38.4 kg, initial speed 10.8 m/s, hemispherical head of 60 mm radius. An explicit initial clearance is *not* provided in §3.1.1: the visualization uses 1 mm **only as a display assumption**.

In [ ]:
@dataclass(frozen=True)
class Parameters:
    L: float=0.600
    W: float=0.070
    H: float=0.100
    dx: float=0.005
    horizon_ratio: float=3.0
    support_span: float=0.500
    E_concrete: float=25e9
    nu_concrete: float=0.25
    rho_concrete: float=2400.0
    ft_concrete: float=4.6e6
    fc_concrete: float=60e6
    Gf_concrete: float=105.0
    E_steel: float=200e9
    rho_steel: float=7750.0
    fy_steel: float=482e6
    bar_diameter: float=0.006
    n_bars: int=2
    impactor_mass: float=38.4
    impactor_speed: float=10.8
    impactor_radius: float=0.060
    dt: float=3e-7

p = Parameters()
L, W, H, dx = p.L, p.W, p.H, p.dx
Nx, Ny, Nz = [round(t/dx) for t in (L,W,H)]
assert all(np.isclose(np.array([Nx,Ny,Nz])*dx, [L,W,H]))
assert (Nx, Ny, Nz)==(120,14,20)
print(f'Grid: {Nx} × {Ny} × {Nz} = {Nx*Ny*Nz:,} concrete material points')
print('Horizon =',p.horizon_ratio*dx,'m; reference time step =',p.dt,'s')


## B. Generate concrete material points and verify counts

In [ ]:
x = (np.arange(Nx)+0.5)*dx
y = (np.arange(Ny)+0.5)*dx
z = (np.arange(Nz)+0.5)*dx
X,Y,Z = np.meshgrid(x,y,z,indexing='ij')
xyz = np.column_stack((X.ravel(),Y.ravel(),Z.ravel())).astype(np.float64)
N = len(xyz); Vpoint=dx**3
assert N==33600
assert np.unique(xyz,axis=0).shape[0]==N
assert np.all(xyz.min(axis=0)>0) and np.all(xyz.max(axis=0)<[L,W,H])
assert np.isclose(N*Vpoint,L*W*H)
print(f'PASS: {N:,} unique points; volume {N*Vpoint:.7f} m³ (target {L*W*H:.7f} m³)')
print('Coordinate min:',xyz.min(axis=0),'max:',xyz.max(axis=0))


## C. Supports, reinforcement, and impactor — distinguish sourced from assumed

The following arrays are **geometry markers**, not applied contact constraints or a working PDROD model. We do not yet apply bond coupling, ground reactions, or impact forces.

In [ ]:
support_x = np.array([(p.L-p.support_span)/2,(p.L+p.support_span)/2])  # DERIVED
bar_z = 0.010                          # ASSUMED from illustrative 80 mm dimension, verify
bar_y = np.array([0.020,0.050])        # ASSUMED (not dimensioned in source)
bar_x = np.array([0.020,p.L-0.020])     # ASSUMED: rod termination / anchorage length
bar_x_nodes = np.linspace(bar_x[0],bar_x[1],round((bar_x[1]-bar_x[0])/dx)+1)
steel_xyz=np.array([[xx,yy,bar_z] for yy in bar_y for xx in bar_x_nodes])
impact_xy=np.array([p.L/2,p.W/2])
illustrative_gap=0.001                 # ASSUMED only for illustration
impact_center=np.array([impact_xy[0],impact_xy[1],p.H+p.impactor_radius+illustrative_gap])
print('Support x positions:',support_x,'m; spacing:',support_x[1]-support_x[0],'m')
print('ASSUMED transverse bar centers:',bar_y,'m; ASSUMED bar depth z:',bar_z,'m')
print('ASSUMED rod node count:',len(steel_xyz))
print('Illustrative initial sphere center:',impact_center,'m')
assert np.isclose(np.diff(support_x)[0],0.5)
assert len(bar_y)==p.n_bars
assert np.all((bar_y>0)&(bar_y<W))
assert bar_z>p.bar_diameter/2


## D. Figures — top and side views plus cross-section

The markers show where constraints and rods must be implemented later; **they are not yet mechanical boundary conditions**.

In [ ]:
fig,axs=plt.subplots(3,1,figsize=(11,10),constrained_layout=True)
# Side view
a=axs[0]; a.add_patch(plt.Rectangle((0,0),L,H,facecolor='#dae9f2',edgecolor='black'))
for sx in support_x: a.plot(sx,-0.007,marker='^',color='#c45a25',ms=12)
a.plot([bar_x[0],bar_x[-1]],[bar_z,bar_z],color='#a53030',lw=3,label='steel rod height (assumed)')
theta=np.linspace(0,2*np.pi,300);a.plot(impact_center[0]+p.impactor_radius*np.cos(theta),impact_center[2]+p.impactor_radius*np.sin(theta),'k--',lw=1.2,label='spherical head projection')
a.set(xlim=(-0.02,0.62),ylim=(-0.025,0.23),xlabel='x (m)',ylabel='z (m)',title='Side view: beam, supports and impactor (schematic)');a.legend(loc='upper right')
# Top view
a=axs[1];a.add_patch(plt.Rectangle((0,0),L,W,facecolor='#dae9f2',edgecolor='black'))
for yy in bar_y: a.plot(bar_x,[yy,yy],color='#a53030',lw=3)
a.plot(*impact_xy,marker='x',color='black',ms=10,mew=2);a.set(xlim=(-0.02,0.62),ylim=(-0.008,0.078),xlabel='x (m)',ylabel='y (m)',title='Plan view: transverse bar positions are ASSUMED');a.set_aspect('equal',adjustable='box')
# Cross section
a=axs[2];a.add_patch(plt.Rectangle((0,0),W,H,facecolor='#dae9f2',edgecolor='black'))
for yy in bar_y:a.add_patch(plt.Circle((yy,bar_z),p.bar_diameter/2,color='#a53030'))
a.set(xlim=(-0.01,0.08),ylim=(-0.012,0.115),xlabel='y (m)',ylabel='z (m)',title='Cross-section: two Ø6 rods, centers to verify');a.set_aspect('equal')
fig.savefig(OUT/'Bowen_reference_geometry.png',dpi=220);plt.show()


## E. Source-based static bond parameters — diagnostic calculation only

From Chapter 2, Eqs. (2-4), (2-6), (2-8), (2-12): constant micromodulus, tensile linear threshold, critical stretch, and distance-dependent micromodulus. **Critical warning:** Bowen derives the critical stretch using one micromodulus and later introduces a different distance weighting. Fracture-energy consistency with the revised micromodulus must be verified before the solver is considered faithful.

In [ ]:
delta=p.horizon_ratio*dx
s0=p.ft_concrete/p.E_concrete
sc=5*p.Gf_concrete/(6*p.E_concrete*delta*s0)
c_constant=12*p.E_concrete/(np.pi*delta**4)
def c_weighted(r):
    r=np.asarray(r)
    return np.where(r<=delta,72*p.E_concrete/(np.pi*delta**4)*(1-(r/delta)**2)**2,0.0)
print(f'Initial stretch s0 = {s0:.8e}')
print(f'Critical stretch sc = {sc:.8e}')
print(f'Constant micromodulus = {c_constant:.8e} [model units]')
print(f'Weighted c(1 dx) = {float(c_weighted(dx)):.8e} [model units]')
assert 0<s0<sc


## F. Optional peridynamic neighbor topology (no force integration)

The full topology can be built for the original grid. Expect millions of undirected pairs and hundreds of MB of temporary memory. The horizon cutoff convention at exactly 3Δx can affect counts; **check inclusion rules against Bowen's original code**. Default disabled for quick setup validation.

In [ ]:
BUILD_NEIGHBORS = False
if BUILD_NEIGHBORS:
    from scipy.spatial import cKDTree
    tree=cKDTree(xyz)
    pairs=tree.query_pairs(r=delta*(1+1e-12), output_type='ndarray')
    distances=np.linalg.norm(xyz[pairs[:,0]]-xyz[pairs[:,1]],axis=1)
    counts=np.bincount(pairs.reshape(-1),minlength=N)
    print('Undirected pairs:',len(pairs),'max / min / mean neighbors:',counts.max(),counts.min(),counts.mean())
    assert np.max(distances) <= delta*(1+1e-10)
else:
    print('Topology construction skipped. Set BUILD_NEIGHBORS=True after verifying geometry.')


## G. Export the exact *geometric setup*, not simulated results

In [ ]:
metadata={
 'source':'Tu Bowen 2025 thesis, Chapter 3 section 3.1.1 (source-based geometry setup)',
 'parameters':asdict(p), 'number_particles':int(N),'grid_shape':[Nx,Ny,Nz],
 'support_x_m':support_x.tolist(),
 'assumptions':{'bar_y_centers_m':bar_y.tolist(),'bar_z_m':bar_z,
                'bar_x_endpoints_m':bar_x.tolist(),'illustrative_impactor_clearance_m':illustrative_gap,
                'grid_origin':"cell-centered; author's grid origin not explicitly specified",
                'supports':'markers only; no constraint/contact formulation implemented',
                'steel':'marker nodes only; no PDROD mechanics implemented',
                'impactor':'geometry only; no contact implementation'},
 'reference_targets_not_simulated':{'peak_force_N':359800.0,'peak_midspan_deflection_m':0.02270}
}
np.savez_compressed(OUT/'Bowen_Chapter3_geometry.npz', concrete_xyz=xyz,
                    support_x=support_x, steel_marker_xyz=steel_xyz, impactor_center=impact_center)
with (OUT/'Bowen_Chapter3_geometry_metadata.json').open('w',encoding='utf-8') as f: json.dump(metadata,f,indent=2)
print('Saved:',[str(v) for v in OUT.iterdir()])


## H. Readiness gate for Stage 2

**Verified in this notebook:** reference dimensions, original number of concrete material points, material parameter declarations, support span, impactor radius/mass/speed, and theoretical stretch values.

**Not implemented or not verifiable from §3.1.1 alone:** exact locations and mesh of reinforcing rods, full PDROD coupling, volume/surface correction discretization, rigid-projectile dynamics/contact law, exact support constraints, irreversible bond history, DIF sampling/unloading rules, full damage visualization, force/deflection outputs, convergence and comparison to Bowen's Table 3-1.

**Next:** source-trace Chapter 2 constitutive relations, write unit tests for elastic and fracture energies, then implement and verify exact force assembly and contact before running the 33,600-point simulation. Reproduction is an evidence-based verification exercise, not parameter fitting.

---
# Stage 2 — Concrete bond constitutive model: verification before impact
**Source:** Tu Bowen (2025), Chapter 2, Eqs. (2-4)–(2-16), Appendix A.

**Scope:** exact algebraic translation of the stated normal-concrete relations, plus transparent numerical checks. This does **not** yet solve the impact boundary-value problem. The steel coupling, surface corrections, and rigid-hammer dynamics remain Stage 3/4.

**Units:** SI. Bond stretch is dimensionless; stretch rate is s⁻¹; bond-force-density scalar has the usual peridynamic force-density units. Compression is handled as an elastic continuation of the written Eq. (2-5); a compression failure law is **not** claimed from these equations.

**Source uncertainty:** The Appendix A snippet multiplies `s0` and `sc` by `dif` without displaying reset operations. Here the reference thresholds are *recomputed from immutable reference values*, to prevent repeated unintended multiplication.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

EPS_QS=3.0e-6  # thesis Eq. (2-13/14): s^-1
F0=10.0e6     # thesis parameter: Pa

def normal_concrete_difs(stretch_rate, fc=p.fc_concrete):
    """Return DIF_Nt and DIF_E from thesis Eqs. (2-13)-(2-14).

    Positive tensile stretch rate is used; compressive/nonpositive rate gives 1.
    Note: the thesis does not completely specify the numerical treatment of
    negative stretch rates; this is a stated implementation convention.
    """
    rate=np.maximum(np.asarray(stretch_rate,dtype=float),0.0)
    exponent=1.0/(10.0+6.0*fc/F0)
    theta=10.0**(6.0*exponent-2.0)
    ratio=np.maximum(rate/EPS_QS,1.0)
    dif_t=np.where(rate<=EPS_QS,1.0,np.where(rate<=1.0,ratio**exponent,theta*ratio**(1.0/3.0)))
    dif_e=np.where(rate<=EPS_QS,1.0,ratio**0.016)
    return dif_t,dif_e

def thresholds_at_rate(stretch_rate, E=p.E_concrete, ft=p.ft_concrete,
                       Gf=p.Gf_concrete, delta=p.horizon_ratio*p.dx):
    """Thesis Eqs. (2-6), (2-8), (2-15), evaluated from *original* constants."""
    s0_ref=ft/E
    sc_ref=5.0*Gf/(6.0*E*delta*s0_ref)
    dif_t,dif_e=normal_concrete_difs(stretch_rate)
    factor=dif_t/dif_e
    return np.asarray(s0_ref*factor),np.asarray(sc_ref*factor)

def micromodulus(distance, delta=p.horizon_ratio*p.dx, E=p.E_concrete, improved=True):
    """Thesis Eq. (2-12) (improved), or Eq. (2-4) (constant)."""
    distance=np.asarray(distance,dtype=float)
    if improved:
        c=72*E/(np.pi*delta**4)*np.maximum(0.0,1.0-(distance/delta)**2)**2
    else:
        c=np.full_like(distance,12*E/(np.pi*delta**4))
    return np.where((distance>0)&(distance<=delta),c,0.0)

print('Quasistatic s0, sc:',thresholds_at_rate(0.0))
for rate in [0.0,3e-6,1e-3,1.0,10.0,100.0]:
    dft,dfe=normal_concrete_difs(rate)
    a,b=thresholds_at_rate(rate)
    print(f'rate={rate:9.3g}/s: DIF_t={float(dft):7.4f}, DIF_E={float(dfe):7.4f}, s0={float(a):.6e}, sc={float(b):.6e}')


## A. Vectorized bond evolution

A bond's maximum previously attained tensile stretch is retained. After reaching the critical stretch at any step, it stays broken. For an unbroken bond, the force follows Bowen's stated bilinear law. **This is a cohesive bond-force model with a binary broken flag**, not a fully specified unloading/reloading damage law. The thesis does not explicitly provide a thermodynamic unloading rule for partially softened bonds; that gap remains visible here.

In [ ]:
def bond_scalar_force(stretch, stretch_rate, distance, broken=False, improved=True):
    """Return scalar bond force density and irreversible failure flag.

    The demonstration assumes instantaneous elastic/softening envelope forces.
    It does not implement an unstated unloading path for partially softened bonds.
    """
    s,sd,r,d=np.broadcast_arrays(np.asarray(stretch,float),np.asarray(stretch_rate,float),
                                np.asarray(distance,float),np.asarray(broken,bool))
    s0,sc=thresholds_at_rate(sd)
    c=micromodulus(r,improved=improved)
    fail=d | (s>=sc)
    elastic=c*s
    # For positive stretches above s0: linear decrease to zero at sc.
    soft=c*s0*np.clip((sc-s)/np.maximum(sc-s0,1.e-30),0.0,1.0)
    f=np.where(s<=s0,elastic,soft)
    return np.where(fail,0.0,f),fail

# A lightweight state holder so persistent failure cannot heal after unloading.
class BondHistory:
    def __init__(self,n):
        self.broken=np.zeros(n,dtype=bool)
        self.max_stretch=np.full(n,-np.inf)
    def update(self,stretch,rate,length):
        stretch=np.asarray(stretch,dtype=float)
        self.max_stretch=np.maximum(self.max_stretch,stretch)
        f,new_broken=bond_scalar_force(stretch,rate,length,self.broken)
        self.broken |= new_broken
        return f


## B. Verification tests (fail loudly)
Checks: elastic-force slope, continuity at softening onset, zero force after failure, no healing, rate-threshold algebra, and input shape handling. The tests do not validate the entire PD structure.

In [ ]:
delta=p.horizon_ratio*p.dx
r=delta*0.5
s0,sc=map(float,thresholds_at_rate(0.0))
c=float(micromodulus(r))

f,fail=bond_scalar_force(np.array([s0/2,s0,s0+(sc-s0)/2,sc+1.e-7]),0.0,r)
assert np.isclose(f[0],c*s0/2,rtol=1e-12)
assert np.isclose(f[1],c*s0,rtol=1e-12)
assert np.isclose(f[2],c*s0/2,rtol=1e-12)
assert f[3]==0 and fail[-1]
assert not np.any(fail[:3])

h=BondHistory(1)
assert h.update([sc*1.1],[0.0],[r])[0]==0
assert h.update([s0*0.1],[0.0],[r])[0]==0  # no healing
assert h.broken[0]

dft,dfe=normal_concrete_difs(1e-2)
a,b=thresholds_at_rate(1e-2)
assert np.isclose(a/s0,dft/dfe) and np.isclose(b/sc,dft/dfe)
assert np.all(np.asarray(normal_concrete_difs(np.array([0.0,1e-3,10.0]))[0])>=1.0)
assert float(micromodulus(delta*1.1))==0.0
print('PASS: bond elastic slope, softening, irreversible failure, rate factors, vectorized shape.')


## C. Visualize the rate-dependent force laws and DIFs

In [ ]:
rates=[0.,1e-2,1.,100.]
fig,axs=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
sweep=np.linspace(-0.00025,0.0040,400)
for q in rates:
    f,_=bond_scalar_force(sweep,q,r)
    axs[0].plot(sweep*1e3,f/(c*s0),label=f'{q:g} s$^{{-1}}$')
axs[0].set(xlabel='Bond stretch (×10⁻³)',ylabel='Force density / (c s₀)',title='Bowen NC bilinear envelope')
axs[0].legend();axs[0].grid(alpha=.3)
rate_sweep=np.logspace(-7,3,240)
dft,dfe=normal_concrete_difs(rate_sweep)
axs[1].semilogx(rate_sweep,dft,label='DIF tensile strength')
axs[1].semilogx(rate_sweep,dfe,label='DIF elastic modulus')
axs[1].semilogx(rate_sweep,dft/dfe,label='Stretch threshold factor',ls='--')
axs[1].set(xlabel='Tensile stretch rate (s⁻¹)',ylabel='Dynamic factor',title='Bowen normal concrete DIFs')
axs[1].legend();axs[1].grid(alpha=.3)
plt.show()


## D. Check fracture-energy calibration versus improved micromodulus

The thesis derives the critical stretch with constant micromodulus (Eq. 2-7 to 2-8), then introduces a different distance-weighted micromodulus (Eq. 2-12). To test whether those definitions automatically yield identical fracture energy, consider a mode-I planar fracture in a homogeneous infinite 3-D medium. For a bond at radial separation `r`, integrate the bilinear traction with bond stretch; the radial weighting for bonds intersecting a plane is proportional to `r³` (up to the same angular constant for both cases).

This test compares **ratios** of the radial energy integrals at fixed `s0` and `sc` and reports the difference. It does not assert that this idealized infinite-domain integral fully captures Bowen's discretized finite beam, nor does it silently recalibrate the model.

In [ ]:
radii=np.linspace(0,delta,20001)
c_const=micromodulus(radii,improved=False)
c_impr=micromodulus(radii,improved=True)
# Relative fracture energy = integral c(r)*r^3 dr, common s0,sc factors cancel.
I_const=np.trapezoid(c_const*radii**3,radii)
I_impr=np.trapezoid(c_impr*radii**3,radii)
ratio=I_impr/I_const
print(f'Weighted radial integral ratio (improved / constant) = {ratio:.6f}')
print(f'Implied energy scale if Gf=105 N/m calibrated for constant c: {p.Gf_concrete*ratio:.2f} N/m')
print('NOTE: This is an analytic/numerical consistency diagnostic, not a measured beam fracture energy.')
assert 0.0<ratio<5.0


## E. Stage 2 readiness gate

**Source-translated and tested:** Eqs. 2-4, 2-5, 2-6, 2-8, 2-12–2-16, and the Appendix A threshold-ratio logic.

**Still unverified or undecided:** precise treatment of rate sign, elastic-modulus scaling inside force density, damage unloading/reloading, bond-volume corrections, near-surface energy treatment, discrete neighbor quadrature, and high-rate energy consistency. No claims are made about matching Chapter 3's peak force or damage yet.

**Next deliverable:** discrete 3-D force assembly with geometrically consistent family search, equal-and-opposite pair forces, elastic patch test and domain/boundary correction audits, then PDROD.


---
# Stage 3 — Full 3-D family topology and conservative concrete force assembly

**Aim:** numerical reproducibility checks for the concrete-only discrete solver, *not* impact results. Stage 1 and Stage 2 cells above define `p`, `xyz`, `micromodulus`, `thresholds_at_rate`, and `bond_scalar_force`. Run cells from the top.

**Important source distinction:** Bowen gives a distance-dependent micromodulus and surface/volume corrections, but the details of exact code-level quadrature, treatment of symmetric corrections and unloading are not all recoverable from the thesis. The **explicit discretization conventions** used below are therefore labeled implementation choices, not authenticated source equivalence.

The assembly uses unique undirected bonds `(i,j)`; each interaction adds force `+F` to `i` and `−F` to `j`, exactly conserving total discrete internal force. The correction factor below is symmetric, so angular momentum is conserved for central pair forces as well. No support or impactor is present in this test.

In [ ]:
from scipy.spatial import cKDTree
from time import perf_counter
try:
    from numba import njit
    HAVE_NUMBA=True
except ImportError:
    HAVE_NUMBA=False
    def njit(*args,**kwargs):
        if args and callable(args[0]): return args[0]
        return lambda fun: fun

# Full Bowen discretization: 120 x 14 x 20 = 33,600 concrete particles.
# The 1e-10 relative tolerance retains exact-horizon lattice bonds.
BUILD_FULL_NETWORK=True
if BUILD_FULL_NETWORK:
    t0=perf_counter()
    tree=cKDTree(xyz)
    ij=tree.query_pairs(r=delta*(1.0+1e-10),output_type='ndarray').astype(np.int32)
    I,J=ij[:,0].copy(),ij[:,1].copy()
    R0=xyz[J]-xyz[I]
    LEN=np.linalg.norm(R0,axis=1)
    UNIT=R0/LEN[:,None]
    counts=np.bincount(np.concatenate([I,J]),minlength=N)
    print(f'Network: N={N:,} particles, {len(I):,} undirected bonds; neighbors min/mean/max={counts.min()}/{counts.mean():.2f}/{counts.max()}')
    print(f'Build time: {perf_counter()-t0:.2f} s; bond length range {LEN.min()*1e3:.2f}-{LEN.max()*1e3:.2f} mm')
    assert N==33600 and I.size>500000
    assert np.all((LEN>0)&(LEN<=delta*(1.0+1e-9)))
    assert np.all(I<J) and np.all(counts>0)


## A. Pair quadrature and boundary-correction conventions

Volume weight: `V=dx³` for every grid point; to prevent double-counting we integrate each unique pair once and distribute `±c*s*V²` as a physical force. This is a midpoint quadrature, **not** proof of an identical implementation to Bowen.

Partial-horizon bonds are weighted with `w=clip((δ+dx/2−r)/dx,0,1)`. For the chosen `δ=3dx`, this gives full weight for bonds at or below `2.5dx` and half weight at `3dx`. This follows a conventional particle-volume approximation, but the exact correction is an implementation choice to verify against original code.

Surface correction is **off by default**; the optional isotropic factor derived from missing family volume is diagnostic only and should not be presented as Bowen's surface correction without confirmation.

In [ ]:
V=dx**3
# Independent toggles for sensitivity; defaults are conservative documented choices.
USE_PARTIAL_VOLUME=True
USE_SURFACE_CORRECTION=False
volweight=np.clip((delta+dx/2-LEN)/dx,0.0,1.0) if USE_PARTIAL_VOLUME else np.ones_like(LEN)

# Relative neighborhood occupancy, based on weighted discrete interaction counts.
occupancy=np.bincount(I,weights=volweight,minlength=N)+np.bincount(J,weights=volweight,minlength=N)
interior=(xyz[:,0]>delta+dx)&(xyz[:,0]<p.L-delta-dx)&(xyz[:,1]>delta+dx)&(xyz[:,1]<p.W-delta-dx)&(xyz[:,2]>delta+dx)&(xyz[:,2]<p.H-delta-dx)
reference_neighbors=float(np.median(occupancy[interior]))
node_correction=np.ones(N)
if USE_SURFACE_CORRECTION:
    node_correction=np.clip(reference_neighbors/np.maximum(occupancy,1),1.0,3.0)
# Geometric symmetric mean retains equal/opposite bond forces.
PAIR_WEIGHT=volweight*np.sqrt(node_correction[I]*node_correction[J])
print('Interior weighted neighbor reference:',reference_neighbors)
print('Surface correction enabled:',USE_SURFACE_CORRECTION,'; pair quadrature weights:',PAIR_WEIGHT.min(),PAIR_WEIGHT.max())
assert interior.any()


## B. Force assembly and bond history

For a bond with current vector `d=(x_j+u_j)−(x_i+u_i)`, stretch `s=(|d|−|ξ|)/|ξ|`, and rate `s_dot≈(s_t−s_(t−1))/dt`, assemble `F_ij=c(|ξ|) f_envelope(s) V² w_ij n_current`. The physical units follow the peridynamic force-density convention `c` in N/m⁶.

**Crucial modeling gap:** Bowen's thesis does not fully specify unloading/reloading of a *partly softened* bond. This stage evaluates the instantaneous envelope and retains a broken flag. Later comparison with the actual Fortran must decide whether additional scalar damage history is needed.


In [ ]:
@njit
def accumulate_unique_pairs(iidx,jidx, vec, amplitudes, n_particles):
    f=np.zeros((n_particles,3),dtype=np.float64)
    for k in range(len(iidx)):
        i=iidx[k];j=jidx[k]
        fx=amplitudes[k]*vec[k,0];fy=amplitudes[k]*vec[k,1];fz=amplitudes[k]*vec[k,2]
        f[i,0]+=fx;f[i,1]+=fy;f[i,2]+=fz
        f[j,0]-=fx;f[j,1]-=fy;f[j,2]-=fz
    return f

class DiscretePDConcrete:
    def __init__(self, xyz, I, J, lengths, weights):
        self.xyz=xyz;self.I=I;self.J=J;self.lengths=lengths
        self.weights=weights;self.broken=np.zeros(len(I),dtype=np.bool_)
        self.prev_stretch=np.zeros(len(I),dtype=np.float64)
    def evaluate(self, displacement, dt, commit=False, override_rate=None):
        deformed=self.xyz+np.asarray(displacement,float)
        d=deformed[self.J]-deformed[self.I]
        cur=np.linalg.norm(d,axis=1)
        stretch=(cur-self.lengths)/self.lengths
        rate=(stretch-self.prev_stretch)/dt if override_rate is None else np.broadcast_to(override_rate,stretch.shape)
        scalar,failed=bond_scalar_force(stretch,rate,self.lengths,self.broken)
        amp=scalar*(V**2)*self.weights
        direction=d/cur[:,None]
        F=accumulate_unique_pairs(self.I,self.J,direction,amp,len(self.xyz))
        if commit:
            self.broken |= failed
            self.prev_stretch[:]=stretch
        return F,stretch,failed
    def damage(self):
        # Per-node unweighted broken-bond fraction; different from weighted material-point damage in thesis.
        return np.bincount(np.r_[self.I,self.J],weights=np.r_[self.broken,self.broken],minlength=len(self.xyz))/counts

model=DiscretePDConcrete(xyz,I,J,LEN,PAIR_WEIGHT)
print('Ready:',len(model.I),'bonds; Numba:',HAVE_NUMBA)


## C. Required numerical verification tests

**Test 1:** undeformed forces vanish. **Test 2:** arbitrary displacement produces equal/opposite total internal forces and near-zero net internal torque. **Test 3:** small homogeneous stretch produces no significant interior force away from boundaries (patch-test-like diagnostic; not a full continuum elastic modulus calibration). **Test 4:** a failed bond cannot heal.


In [ ]:
import numpy.testing as npt
t0=perf_counter()
zero=np.zeros_like(xyz)
Fzero,s,broken=model.evaluate(zero,p.dt)
assert np.max(np.abs(Fzero))<1e-7
print('PASS 1: undeformed state has zero internal force')

# Nonuniform, smooth displacement: small relative to stretching threshold.
ux=1.e-6*np.sin(np.pi*xyz[:,0]/p.L)*np.sin(np.pi*xyz[:,2]/p.H)
u=np.column_stack([ux,0.4*ux,-0.2*ux])
F,s,broken=model.evaluate(u,p.dt,override_rate=0.0)
net=F.sum(axis=0)
scale=np.sum(np.linalg.norm(F,axis=1))
assert np.linalg.norm(net) <= max(1.e-8, 1.e-10*scale)
# Central forces imply vanishing internal torque about arbitrary fixed origin.
torque=np.sum(np.cross(xyz+u,F),axis=0)
assert np.linalg.norm(torque)<max(1.e-8, 1.e-9*scale*p.L)
print(f'PASS 2: net internal force {np.linalg.norm(net):.3e} N, torque {np.linalg.norm(torque):.3e} N m')

# Uniform infinitesimal isotropic dilation, below first damage threshold.
eps=1.e-5
u_aff=eps*(xyz-np.array([p.L/2,p.W/2,p.H/2]))
Faff,saff,baff=model.evaluate(u_aff,p.dt,override_rate=0.0)
force_mag=np.linalg.norm(Faff,axis=1)
print('Interior affine-force max [N]:',force_mag[interior].max(),'boundary max [N]:',force_mag[~interior].max())
assert force_mag[interior].max()<1.e-7
assert not np.any(baff)
print('PASS 3: affine-strain interior force cancellation')

# Fracture test: force remains zero even after unloading to intact stretch.
# Shift one corner particle far away, mark all its connected bonds failed.
corner=0
u_break=np.zeros_like(xyz); u_break[corner]=[0.08,0.0,0.0]
Fb,sb,bb=model.evaluate(u_break,p.dt,commit=True,override_rate=0.0)
assert np.any(model.broken)
broken_before=model.broken.copy()
Fu,su,bu=model.evaluate(zero,p.dt,commit=True,override_rate=0.0)
assert np.all(model.broken[broken_before])
assert np.max(np.abs(Fu))<1.e-7
print('PASS 4: damage remains after unloading;',broken_before.sum(),'bonds broken')
# Reset history before any physical simulation.
model.broken[:]=False;model.prev_stretch[:]=0.0
print(f'All Stage 3 assembly tests passed. Elapsed {perf_counter()-t0:.2f} s')


## D. Characterize neighbor and correction sensitivity

Plots below are diagnostics of the discrete force network, not the crack pattern of an impacted RC beam.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
midz=np.abs(xyz[:,2]-p.H/2)<dx/2+1e-10
sc=axs[0].scatter(xyz[midz,0]*1e3,xyz[midz,1]*1e3,c=counts[midz],s=5,cmap='viridis')
axs[0].set(xlabel='Beam length x (mm)',ylabel='Width y (mm)',title='Peridynamic family size — middle layer')
fig.colorbar(sc,ax=axs[0],label='Neighbor count')
axs[1].hist(LEN/dx,bins=np.arange(0.5,3.51,0.1),edgecolor='none')
axs[1].set(xlabel='Bond length / particle spacing',ylabel='Bond frequency',title='Full 3-D bond-length distribution')
plt.show()


## E. Stage 3 conclusion and remaining fidelity checks

The network and force assembly are **fully operational** for the concrete-only geometry; uniform reference state, pairwise conservation, interior affine-force cancellation, and nonhealing damage are tested.

This stage is **not** an experimental validation or exact Bowen source reproduction. Before dynamic loading we must verify: (1) actual Fortran volume/surface factors, (2) the distance-dependent micromodulus elastic calibration on this mesh, (3) unloading laws, (4) stage-dependent DIF updates, (5) support implementation, (6) exact PDROD interpolation, and (7) rigid-hammer mass/contact integrator.

**Next stage: PDROD coupling verification** against Bowen's 2-D reinforced tensile-plate and four-point-bending demonstrations, followed by impact/contact setup.

---
# Stage 4 — PDROD eight-node concrete interpolation and rod elements

**Source:** Tu Bowen thesis §2.5.1, Eqs. (2-23)–(2-37), Appendix C. Appendix C interpolates the displacement of a steel–concrete coupling location from eight concrete displacements, and applies the opposite coupling force to those concrete nodes using the eight interpolation coefficients.

**Important fidelity limitation:** The published excerpt does not uniquely fix the layout/number of coupling locations, the exact longitudinal-bar anchorage, or all concrete–steel interface constants. We use the Stage 1 *provisional* rod placement, a transparent set of interface samples, and a tunable coupling stiffness. This verifies a conservative three-dimensional discretization but **does not authenticate Bowen's exact PDROD parameterization**. In the thesis body, 2-D four-node bilinear shape functions are illustrated; Appendix C uses eight nodes, for which we use standard 3-D trilinear weights.

No hammer, supports, full motion integration or claimed match to the Chapter 3 force history is included.

In [ ]:
# Standalone Stage 4 inputs from earlier cells: xyz, dx, p, steel_xyz, bar_x_nodes, bar_y, V.
steel_xyz=np.asarray(steel_xyz,dtype=np.float64)
Nsteel=len(steel_xyz)
rod_pairs=np.concatenate([np.column_stack([np.arange(k*len(bar_x_nodes),(k+1)*len(bar_x_nodes)-1),
                                            np.arange(k*len(bar_x_nodes)+1,(k+1)*len(bar_x_nodes))])
                          for k in range(len(bar_y))]).astype(np.int32)
steel_E=200e9
steel_diameter=0.006  # nominal Chapter 3 reinforcement; confirm against original figure
steel_A=np.pi*steel_diameter**2/4
steel_lengths=np.linalg.norm(steel_xyz[rod_pairs[:,1]]-steel_xyz[rod_pairs[:,0]],axis=1)
assert np.all(steel_lengths>0)
print('Steel nodes:',Nsteel,'rod elements:',len(rod_pairs),'nominal area [mm²]:',steel_A*1e6)


## A. Exact-to-grid trilinear interpolation

The eight surrounding concrete particles are found from the Cartesian concrete **cell-center** grid. If an interpolation point lies exactly on a mesh plane, its weights are still nonnegative and sum to one. For arbitrary positions inside the center-grid bounding box, the map reproduces constants and affine fields exactly. Coupling samples outside this region are rejected, not clipped without warning.

In [ ]:
def concrete_grid_eight(point):
    point=np.asarray(point,dtype=float)
    nx,ny,nz=Nx,Ny,Nz
    h=dx
    a=point/h-0.5
    base=np.floor(a).astype(int)
    if np.any(base<0) or np.any(base>=np.array([nx-1,ny-1,nz-1])):
        raise ValueError(f'Coupling point {point} outside center-grid interpolation domain')
    frac=a-base
    inds=[]; ws=[]
    for qx in (0,1):
        for qy in (0,1):
            for qz in (0,1):
                inds.append(np.ravel_multi_index((base[0]+qx,base[1]+qy,base[2]+qz),(nx,ny,nz)))
                ws.append((frac[0] if qx else 1-frac[0])*(frac[1] if qy else 1-frac[1])*(frac[2] if qz else 1-frac[2]))
    return np.array(inds,dtype=np.int32),np.array(ws,dtype=float)

# Couple steel nodes at their current locations to interpolated concrete kinematic points,
# with an optionally offset sampling location to avoid a zero-length bond.
# A transverse offset on each rod side allows a finite interface bond direction.
COUPLING_OFFSET=dx      # a provisional geometric choice, not established from Bowen's source
COUPLING_STIFFNESS=2.0e8 # N/m per coupling location: provisional, not calibrated
sample_steel=[];sample_pos=[];sample_idx=[];sample_w=[]
for si, pos in enumerate(steel_xyz):
    for sign in (-1.,1.):
        pt=pos+np.array([0.,sign*COUPLING_OFFSET,0.])
        idx,w=concrete_grid_eight(pt)
        sample_steel.append(si);sample_pos.append(pt);sample_idx.append(idx);sample_w.append(w)
sample_steel=np.asarray(sample_steel,dtype=np.int32)
sample_pos=np.asarray(sample_pos,dtype=float)
sample_idx=np.asarray(sample_idx,dtype=np.int32)
sample_w=np.asarray(sample_w,dtype=float)
assert np.all(sample_w>=-1e-14)
np.testing.assert_allclose(sample_w.sum(axis=1),1.,atol=1e-13)
np.testing.assert_allclose((xyz[sample_idx]*sample_w[:,:,None]).sum(axis=1),sample_pos,atol=1e-13)
print('Coupling samples:',len(sample_steel),'eight-node maps:',sample_idx.shape)


## B. Steel axial element forces and conservative interface force transfer

Linear axial rods use the classical 3-D truss stiffness \(EA/L\). For the coupling we use a *vectorial penalty spring* \(\mathbf f_s=k_c(\mathbf u_c-\mathbf u_s)\) where \(\mathbf u_c=\sum_{a=1}^8N_a\mathbf u_a\). The concrete receives \(-N_a\mathbf f_s\), so overall force and virtual work balance exactly.

This spring is a **numerical verification surrogate** for Bowen's bond-stretch-dependent constitutive force; its constant stiffness must be replaced by the Appendix C bond law for faithful reproduction. In particular, the stage does not yet model interface slip, fracture, or strain-rate effects.

In [ ]:
def rod_forces(ust):
    us=np.asarray(ust,float)
    d0=steel_xyz[rod_pairs[:,1]]-steel_xyz[rod_pairs[:,0]]
    unit=d0/steel_lengths[:,None]
    ext=np.sum((us[rod_pairs[:,1]]-us[rod_pairs[:,0]])*unit,axis=1)
    fs=(steel_E*steel_A/steel_lengths*ext)[:,None]*unit
    out=np.zeros_like(us)
    np.add.at(out,rod_pairs[:,0],fs)
    np.add.at(out,rod_pairs[:,1],-fs)
    return out

def coupling_forces(ucon,ust,k=COUPLING_STIFFNESS):
    uc=np.sum(np.asarray(ucon)[sample_idx]*sample_w[:,:,None],axis=1)
    rel=uc-np.asarray(ust)[sample_steel]
    # force ON steel and its 8-concrete-node reaction
    fsteel_sample=k*rel
    fs=np.zeros_like(ust)
    fc=np.zeros_like(ucon)
    np.add.at(fs,sample_steel,fsteel_sample)
    for a in range(8):
        np.add.at(fc,sample_idx[:,a],-sample_w[:,a,None]*fsteel_sample)
    return fc,fs


## C. Automated coupling verification tests

Tests cover shape functions, affine reproduction, steel axial equilibrium, coupling action–reaction, moment consistency, and virtual work. Results are algorithm checks, not experimental validation.

In [ ]:
rng=np.random.default_rng(2026)
uc=np.zeros_like(xyz);us=np.zeros_like(steel_xyz)
fc,fs=coupling_forces(uc,us)
assert np.max(np.abs(fc))==0 and np.max(np.abs(fs))==0
print('PASS 1: undeformed interface has zero force')

# Common translation of steel and concrete: no artificial coupling traction.
translation=np.array([1.1e-5,-1.5e-5,2.0e-5])
fc,fs=coupling_forces(np.tile(translation,(N,1)),np.tile(translation,(Nsteel,1)))
assert np.linalg.norm(fc)+np.linalg.norm(fs)<1e-6
print('PASS 2: rigid translation gives zero coupling force')

# Affine displacement reproduced exactly by interpolation.
A=np.array([[1e-5,2e-6,0],[3e-6,-1e-5,1e-6],[0,0,2e-5]])
ucon=xyz@A.T; usteel=steel_xyz@A.T
interpolated=np.sum(ucon[sample_idx]*sample_w[:,:,None],axis=1)
np.testing.assert_allclose(interpolated,sample_pos@A.T,rtol=0,atol=5e-17)
print('PASS 3: trilinear interpolation exactly reproduces affine displacement')

# Net force, moment and discrete virtual work for arbitrary (small) displacements.
ucon=rng.normal(0,1e-7,xyz.shape); usteel=rng.normal(0,1e-7,steel_xyz.shape)
fc,fs=coupling_forces(ucon,usteel)
net=fc.sum(axis=0)+fs.sum(axis=0)
scale=np.sum(np.abs(fc))+np.sum(np.abs(fs))
assert np.linalg.norm(net)<1e-9*max(scale,1)
# Noncoincident coupling bonds do exert a physical couple if the chosen force is not central.
# The identity checked is the work conjugacy of interpolation/force transfer, not zero net torque.
ducon=rng.normal(0,1e-8,xyz.shape); dusteel=rng.normal(0,1e-8,steel_xyz.shape)
lhs=np.sum(fc*ducon)+np.sum(fs*dusteel)
rel=np.sum(ucon[sample_idx]*sample_w[:,:,None],axis=1)-usteel[sample_steel]
drel=np.sum(ducon[sample_idx]*sample_w[:,:,None],axis=1)-dusteel[sample_steel]
rhs=np.sum(COUPLING_STIFFNESS*rel*drel)
np.testing.assert_allclose(lhs,-rhs,rtol=1e-10,atol=1e-12)
print('PASS 4: coupling action/reaction and virtual work:',np.linalg.norm(net),'N')

fr=rod_forces(usteel)
assert np.linalg.norm(fr.sum(axis=0))<1e-8
print('PASS 5: rod axial forces are equal/opposite; net force:',np.linalg.norm(fr.sum(axis=0)),'N')
print('STAGE 4 UNIT TESTS PASSED (not impact validation).')


## D. Reinforcement geometry and interpolation support diagnostics

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
axs[0].scatter(xyz[::30,0]*1e3,xyz[::30,2]*1e3,s=2,alpha=.12,label='Concrete particles')
for iy,yy in enumerate(bar_y):
    q=steel_xyz[np.abs(steel_xyz[:,1]-yy)<1e-10]
    axs[0].plot(q[:,0]*1e3,q[:,2]*1e3,lw=2,label=f'Steel rod {iy+1}')
axs[0].set(xlabel='Beam x (mm)',ylabel='Beam z (mm)',title='Provisional reinforcement layout')
axs[0].legend()
axs[1].hist(sample_w.ravel(),bins=np.linspace(0,1,26))
axs[1].set(xlabel='Eight-node interpolation weight',ylabel='Frequency',title='Interface interpolation weights')
plt.show()


## Stage 4 result and next verification gate

**Implemented:** 3-D eight-node interpolation aligned with Appendix C's explicit eight-node distribution, linear rod axial mechanics from Eq. (2-31), conservative steel–concrete force transfer, tests of kinematic reproduction and virtual work.

**Not yet source-identical:** exact coupling point generation, steel embedding coordinates, interface spring law and mass assignment. Note the distinction between the paper's illustrated 2-D 4-node interpolation and the eight-node 3-D code appendix. These must be resolved before the Chapter 3 dynamic simulation. **Next:** replace provisional penalty interface with Appendix C's rate/bond-history law; verify rod masses and source supports; then implement rigid falling hammer and explicit time integration.

---
# Stage 5 — Hammer contact, support constraints and explicit dynamics

**Purpose:** Add a dynamic integrator and a movable rigid **hemispherical/spherical tip** to the verified Stage 4 mechanics. The concrete beam keeps the Chapter 3 600×70×100 mm, 33,600-point model. The hammer has an assigned 38.4 kg mass and 10.8 m/s initial downward velocity. **This is not yet an exact reproduction of Bowen's contact algorithm.** The contact is a documented penalty approximation pending recovery of Bowen's constraint/contact and steel-interface implementations.

**IMPORTANT:** `RUN_DYNAMIC = False` by default. A contact-only verification test executes immediately. To simulate the beam, set `RUN_DYNAMIC = True` after reviewing assumptions. The initial beam/hammer gap and steel–concrete interface stiffness remain **provisional**. Outputs must not be represented as reproduced Chapter 3 force/deflection results.


## A. Prescribed geometry, provisional contact, and support masks

In [ ]:
import time
# Reconstructed Chapter 3 impact configuration (source: thesis).
HAMMER_MASS = p.impactor_mass
HAMMER_RADIUS = p.impactor_radius
HAMMER_V0 = -p.impactor_speed
# Assumed clearance to beam TOP envelope; top particle layer is z=H-dx/2.
INITIAL_GAP = 0.001  # m; provisional, not specified as an exact number by thesis
hammer_z_initial = float(np.max(xyz[:,2]) + HAMMER_RADIUS + INITIAL_GAP)
hammer_xy = np.array([p.L/2,p.W/2])
# Stiffness is an explicitly provisional penalty; diagnose dt and resulting penetration.
CONTACT_STIFFNESS = 2.0e8  # N/m PER PARTICLE; not Bowen's exact contact model
CONTACT_DAMPING = 0.0  # N s/m, set zero to expose energy drift
USE_GRAVITY = False    # constant impact velocity is imposed as INITIAL condition, not prescribed motion
# Support patches on the underside, at x = 50 and 550 mm, spanning beam width.
support_at_x = np.min(np.abs(xyz[:,0,None]-support_x[None,:]),axis=1) <= dx/2 + 1e-12
support_at_z = xyz[:,2] <= dx/2+1e-12
support_mask = support_at_x & support_at_z
SUPPORT_DOF_Z_ONLY=True  # roller-like vertical supports at BOTH supports: provisional simplification
# To eliminate horizontal rigid-body drift, fix x motion for ONE support particle and y for another.
anchor_candidates = np.flatnonzero(support_mask)
assert len(anchor_candidates) >= 2
anchor_x = anchor_candidates[0]; anchor_y = anchor_candidates[1]
print('Hammer center (initial):',hammer_z_initial,'m; initial gap:',INITIAL_GAP*1e3,'mm')
print('Support constrained bottom-layer particles:',int(support_mask.sum()))
print('Provisional particle contact stiffness:',CONTACT_STIFFNESS,'N/m')
assert abs(HAMMER_V0)==10.8 and len(anchor_candidates)>0


## B. Conservative sphere–material-point contact

The sphere and slab exchange exactly opposing forces. Contact is activated only for **penetration** of concrete point positions into the current sphere. Hammer is allowed to recoil and slow down; it is not driven by a prescribed moving displacement. The contact response is *a numerical penalty regularization*, not the rigid projection used in some PD impact studies. Results depend on contact stiffness, point spacing and contact radius, all requiring independent convergence.

In [ ]:
def sphere_contact(xyz_def, hammer_center_z, v_concrete=None, vz_hammer=0.0, k=CONTACT_STIFFNESS):
    center=np.array([hammer_xy[0],hammer_xy[1],hammer_center_z],dtype=float)
    d=xyz_def-center[None,:]
    distances=np.linalg.norm(d,axis=1)
    penetration=np.maximum(HAMMER_RADIUS-distances,0.)
    active=penetration>0
    forces=np.zeros_like(xyz_def)
    if active.any():
        unit=d[active]/distances[active,None]
        magnitudes=k*penetration[active]
        forces[active]=magnitudes[:,None]*unit
    f_hammer = -np.sum(forces,axis=0)
    potential=0.5*k*np.sum(penetration[active]**2)
    return forces, f_hammer, float(potential),int(active.sum()),float(penetration.max(initial=0.0))

# Standalone test: one point 0.25 mm inside the sphere surface, and out-of-contact configuration.
point=np.array([[p.L/2,p.W/2,0.0]])
f0,h0,E0,n0,pen0=sphere_contact(point,100.0)
assert n0==0 and np.all(f0==0) and np.all(h0==0)
cz=HAMMER_RADIUS - 0.00025
f1,h1,E1,n1,pen1=sphere_contact(point,cz)
assert n1==1 and f1[0,2]<0 and h1[2]>0
np.testing.assert_allclose(f1.sum(axis=0)+h1,0.0,atol=1e-10)
np.testing.assert_allclose(E1,0.5*CONTACT_STIFFNESS*pen1**2,rtol=1e-9)
print('PASS: hammer/contact forces are equal and opposite; positive repulsion and potential verified')
print('Single-point penetration (mm):',pen1*1e3,'; force (kN):',abs(h1[2])/1e3)


## C. Separate concrete, reinforcement and hammer masses

In [ ]:
CONCRETE_NODE_MASS = p.rho_concrete*dx**3
# Lump half of each element's mass to each adjacent steel node.
STEEL_MASS = np.zeros(Nsteel)
rod_mass_per_element=p.rho_steel*steel_A*steel_lengths
np.add.at(STEEL_MASS,rod_pairs[:,0],0.5*rod_mass_per_element)
np.add.at(STEEL_MASS,rod_pairs[:,1],0.5*rod_mass_per_element)
assert np.all(STEEL_MASS>0)
print('Concrete mass (kg):',N*CONCRETE_NODE_MASS)
print('Steel mass (kg):',STEEL_MASS.sum())
print('Hammer mass (kg):',HAMMER_MASS)
print('Initial hammer kinetic energy (J):',0.5*HAMMER_MASS*HAMMER_V0**2)


## D. Explicit velocity-Verlet integrator — research prototype

The solver updates concrete, rods and hammer at every step. It records contact force, midspan deflection, reaction force, bond failure and energy diagnostics. Enforced support nodes have zero vertical displacement and velocity. The horizontal anchors eliminate free rigid-body drift. Steel–concrete penalty interface is inherited unchanged from Stage 4 and must be replaced for *exact* PDROD replication.

**Energy diagnostic caveat:** With strain-rate-dependent softening and bond-breaking, a scalar elastic strain-energy expression is insufficient to establish full energy conservation. We therefore report kinetic, contact potential, a crude work-based internal energy accumulator, and *apparent* balance as diagnostics, **not as a validated thermodynamic energy accounting**.


In [ ]:
def apply_supports(u,v):
    u[support_mask,2]=0.0
    v[support_mask,2]=0.0
    u[anchor_x,0]=0.0; v[anchor_x,0]=0.0
    u[anchor_y,1]=0.0; v[anchor_y,1]=0.0

def system_forces(uc, us, zh, dt, commit=False):
    fpd,stretch,failed=model.evaluate(uc,dt,commit=commit)
    fc,fs=coupling_forces(uc,us)
    fs+=rod_forces(us)
    cforce,hforce,epot,ncontact,maxpenetration=sphere_contact(xyz+uc,zh)
    fc+=fpd+cforce
    return fc,fs,hforce[2],epot,ncontact,maxpenetration,int(failed.sum())

def reset_concrete_history():
    model.broken[:]=False
    model.prev_stretch[:]=0.

def run_impact(num_steps=100,save_stride=10,dt=p.dt,verbose=True):
    reset_concrete_history()
    uc=np.zeros_like(xyz); vc=np.zeros_like(xyz)
    us=np.zeros_like(steel_xyz); vs=np.zeros_like(steel_xyz)
    zh=hammer_z_initial; vh=HAMMER_V0
    results=[]
    fcon,fsteel,fh,ec,ncontact,pen,nfailed=system_forces(uc,us,zh,dt,commit=True)
    acon=fcon/CONCRETE_NODE_MASS
    asteel=fsteel/STEEL_MASS[:,None]
    ah=(fh/HAMMER_MASS)-(9.81 if USE_GRAVITY else 0.)
    t0=time.perf_counter()
    midspan=(np.abs(xyz[:,0]-p.L/2)<=dx/2+1e-12) & (xyz[:,2]>p.H/2)
    for step in range(1,num_steps+1):
        vc+=0.5*dt*acon;vs+=0.5*dt*asteel;vh+=0.5*dt*ah
        uc+=dt*vc;us+=dt*vs;zh+=dt*vh
        apply_supports(uc,vc)
        # Recalculate forces with state at the NEW step.
        fcon,fsteel,fh,ec,ncontact,pen,nfailed=system_forces(uc,us,zh,dt,commit=True)
        acon=fcon/CONCRETE_NODE_MASS
        asteel=fsteel/STEEL_MASS[:,None]
        ah=(fh/HAMMER_MASS)-(9.81 if USE_GRAVITY else 0.)
        vc+=0.5*dt*acon;vs+=0.5*dt*asteel;vh+=0.5*dt*ah
        apply_supports(uc,vc)
        if step%save_stride==0 or step==1:
            kinetic=(0.5*CONCRETE_NODE_MASS*np.sum(vc**2)+0.5*np.sum(STEEL_MASS[:,None]*vs**2)+0.5*HAMMER_MASS*vh**2)
            rec=dict(step=step,time_s=step*dt,hammer_z=zh,hammer_v=vh,
                     impact_force_kN=max(fh,0.0)/1e3,contact_points=ncontact,
                     penetration_mm=pen*1e3,
                     midspan_deflection_mm=-float(np.mean(uc[midspan,2]))*1e3,
                     max_displacement_mm=float(np.max(np.linalg.norm(uc,axis=1)))*1e3,
                     broken_bonds=int(model.broken.sum()),kinetic_J=kinetic,contact_potential_J=ec)
            results.append(rec)
            if verbose and (step==1 or step%max(save_stride,50)==0):
                print(f"{step:6d}/{num_steps}: force={rec['impact_force_kN']:.3f} kN; "
                      f"deflection={rec['midspan_deflection_mm']:.5f} mm; "
                      f"contacts={ncontact}; broken={rec['broken_bonds']}")
        if not (np.isfinite(uc).all() and np.isfinite(us).all() and np.isfinite(zh)):
            raise FloatingPointError(f'Nonfinite state at step {step}')
    if verbose:print(f'Elapsed {time.perf_counter()-t0:.2f}s')
    return results,(uc,vc,us,vs,zh,vh)


## E. Explicit run switch and illustrative visualization

The default test is intentionally **not** the 100,000-step impact calculation. At 10.8 m/s and the assumed 1 mm clearance to the upper particle layer, first contact is expected at approximately `INITIAL_GAP / 10.8`, i.e., around 309 time steps. If a run ends before this, zero contact forces are expected.


In [ ]:
RUN_DYNAMIC=False      # Set True only when ready to run full model
DYNAMIC_STEPS=400      # Initial trial reaches contact if solver remains stable
SAVE_EVERY=10
if RUN_DYNAMIC:
    impact_records,impact_state=run_impact(num_steps=DYNAMIC_STEPS,save_stride=SAVE_EVERY)
    tt=np.array([r['time_s'] for r in impact_records])*1e3
    fig,ax=plt.subplots(2,1,figsize=(10,7),sharex=True,constrained_layout=True)
    ax[0].plot(tt,[r['impact_force_kN'] for r in impact_records]);ax[0].set_ylabel('Contact force (kN)')
    ax[1].plot(tt,[r['midspan_deflection_mm'] for r in impact_records]);ax[1].set_ylabel('Top-midspan deflection (mm)')
    ax[1].set_xlabel('Time (ms)')
    plt.show()
    print('PROTOTYPE peak force (kN)',max(r['impact_force_kN'] for r in impact_records))
    print('PROTOTYPE peak deflection (mm)',max(r['midspan_deflection_mm'] for r in impact_records))
else:
    print('Full 3-D dynamics NOT RUN. Set RUN_DYNAMIC=True after reviewing assumptions.')
    print('Expected first-contact step:',int(np.ceil(INITIAL_GAP/(abs(HAMMER_V0)*p.dt))))


## F. Readiness gate for final Bowen Chapter 3 comparison

**Implemented and unit-tested:** explicit 3-D rigid-body hammer degree of freedom, contact-only force and potential checks, concrete mass, rod mass, support masks and time-stepping software pathway.

**Not yet source-equivalent or validated:** (1) exact eight-node nonlinear PDROD interaction / irreversible interface law; (2) actual support fixture condition; (3) Bowen's rigid impact contact enforcement and treatment of particles; (4) dynamic modulus implementation; (5) contact convergence; (6) physical energy accounting; (7) reproducing Figure 3-3, Figure 3-4 and Table 3-1. These remain research blockers, *not* errors to hide by adjusting penalty stiffness.


---
# Stage 6 — Source-fidelity audit and dynamic readiness checks

**Research status:** The Chapter 3 particle geometry is reproduced, but Stage 5 still contains a penalty-based rod–concrete interaction, a penalty spherical contact law, and provisional supports. The code is a *research reproduction prototype*, not a numerical match to Bowen. This stage adds quantitative regression diagnostics, records a machine-readable audit, and generates a small non-contact-to-contact diagnostic only if explicitly enabled.

**Do not calibrate by forcing the reported 359.8 kN and 22.70 mm to match.** First obtain the missing source-equivalent algorithms, demonstrate energy accounting, and establish grid/contact convergence.


In [ ]:
from pathlib import Path
import csv, json
from datetime import datetime

STAGE6_DIR=Path('bowen_stage6_outputs')
STAGE6_DIR.mkdir(exist_ok=True)
TARGET_PEAK_FORCE_KN=359.8
TARGET_PEAK_DEFLECTION_MM=22.70
REPRODUCTION_AUDIT={
    'beam_geometry': ('verified_in_notebook','600 x 70 x 100 mm; 33,600 material points'),
    'bond_force_assembly': ('verified_unit_tests','pairwise momentum conservation'),
    'microelastic_fracture': ('partly_verified','continuous radial fracture energy checked, discrete fracture energy pending'),
    'strain_rate_DIF': ('implemented_not_source_equivalent','DIF_E force-law scaling and update order require original code'),
    'PDROD': ('prototype_only','2 samples per steel node; linear penalty, not thesis nonlinear interface'),
    'contact': ('prototype_only','per-particle spherical penalty; Bowen contact resolution not established'),
    'supports': ('provisional','two point-row vertical masks; fixture agreement not established'),
    'hammer_motion': ('implemented','free rigid hammer with v0 = -10.8 m/s'),
    'energy': ('incomplete','no validated elastic+fracture+rate dissipation balance'),
    'experimental_comparison': ('not_attempted','no full benchmark computation verified')
}
with (STAGE6_DIR/'source_fidelity_audit.csv').open('w',newline='') as f:
    w=csv.writer(f);w.writerow(['component','status','evidence_or_missing_detail'])
    for k,(status,desc) in REPRODUCTION_AUDIT.items():w.writerow([k,status,desc])
print('Audit saved to',STAGE6_DIR/'source_fidelity_audit.csv')
for k,(status,_) in REPRODUCTION_AUDIT.items():print(f'{k:24s}: {status}')


## A. Contact geometry and approach test

At the initial time the sphere's lowest tip is one millimetre above the **highest material-point layer**. As the indenter approaches, lateral offset of concrete particles means the first sphere–point contact may lag the nominal tip-clearance time. The following geometric search identifies the earliest kinematic contact of the sphere with any *undeformed* material point (no impact force calculation).

In [ ]:
center_xy=hammer_xy
radial2=(xyz[:,0]-center_xy[0])**2+(xyz[:,1]-center_xy[1])**2
eligible=radial2<HAMMER_RADIUS**2
# For a sphere above the beam, the first point-contact height is z_i+sqrt(R^2-r^2).
first_center_height=np.max(xyz[eligible,2]+np.sqrt(HAMMER_RADIUS**2-radial2[eligible]))
travel=max(hammer_z_initial-first_center_height,0.0)
first_t=travel/abs(HAMMER_V0)
first_step=int(np.ceil(first_t/p.dt))
print(f'First geometric contact with undeformed points: t={first_t*1e6:.3f} microseconds; step {first_step}')
print(f'First-point equivalent travel: {travel*1e3:.5f} mm')
assert first_step>=1
Fc0,Fh0,Uc0,n0,pen0=sphere_contact(xyz,hammer_z_initial)
assert n0==0 and np.linalg.norm(Fh0)==0.0
print('PASS: initial contact is zero; predicted geometric first-contact step computed')


## B. Discrete contact potential / force consistency

The provisional penalty force must be the negative gradient of the discrete contact potential with respect to the *concrete coordinates* (except at the non-smooth entry boundary). This test confirms the energy derivative for a particle penetrating at a general oblique contact position.

In [ ]:
pt=np.array([[p.L/2+0.005, p.W/2+0.002, 0.0]])
z_test=HAMMER_RADIUS-0.002
fcont,fh,ep,nn,dp=sphere_contact(pt,z_test)
assert nn==1
for d in range(3):
    eps=1e-7
    pplus=pt.copy();pminus=pt.copy()
    pplus[0,d]+=eps;pminus[0,d]-=eps
    plus=sphere_contact(pplus,z_test)[2]
    minus=sphere_contact(pminus,z_test)[2]
    grad=(plus-minus)/(2*eps)
    np.testing.assert_allclose(fcont[0,d],-grad,atol=1e-3,rtol=1e-5)
print('PASS: finite-difference contact force equals negative gradient of potential')


## C. Steel–concrete penalty energy conjugacy

The penalty interface in Stage 4 is conservative *when its stiffness is fixed*: \(U_c=	frac12k\sum\|\sum N_a u_a-u_s\|^2\). This test checks its virtual-work balance by finite difference. **This does not validate the penalty as Bowen's PDROD model.**

In [ ]:
rng=np.random.default_rng(641)
uc_a=rng.normal(0,1e-7,size=xyz.shape)
us_a=rng.normal(0,1e-7,size=steel_xyz.shape)

def coupling_potential(uc,us):
    interp=np.sum(uc[sample_idx]*sample_w[:,:,None],axis=1)
    rel=interp-us[sample_steel]
    return .5*COUPLING_STIFFNESS*np.sum(rel*rel)

fc_a,fs_a=coupling_forces(uc_a,us_a)
np.testing.assert_allclose(fc_a.sum(axis=0)+fs_a.sum(axis=0),np.zeros(3),atol=5e-10)
# select an actually mapped concrete material point
ii=int(sample_idx[len(sample_idx)//2,3]);jj=int(sample_steel[len(sample_steel)//2]);eps=1e-8
up=uc_a.copy();um=uc_a.copy();up[ii,2]+=eps;um[ii,2]-=eps
fdc=-(coupling_potential(up,us_a)-coupling_potential(um,us_a))/(2*eps)
np.testing.assert_allclose(fdc,fc_a[ii,2],atol=1e-3,rtol=1e-5)
up=us_a.copy();um=us_a.copy();up[jj,1]+=eps;um[jj,1]-=eps
fds=-(coupling_potential(uc_a,up)-coupling_potential(uc_a,um))/(2*eps)
np.testing.assert_allclose(fds,fs_a[jj,1],atol=1e-3,rtol=1e-5)
print('PASS: interface momentum balance and penalty-potential gradients')


## D. Dynamic reproducibility gate and conditional execution

A complete comparison requires the resolved source-equivalent interface/contact formulation, a convergence study, and a full impact force–time/deflection–time history. Running longer with known provisional mechanisms does not fix source fidelity. You may explicitly activate a **diagnostic** dynamic run. No benchmark claims are made, even if the response happens to approach the target peaks.

In [ ]:
# This is intentionally OFF. Stage 6 tests above run without a costly full dynamic simulation.
RUN_STAGE6_DIAGNOSTIC=False
STAGE6_DIAGNOSTIC_STEPS=first_step+100
STAGE6_SAVE_EVERY=10
if RUN_STAGE6_DIAGNOSTIC:
    dynamic_rows, dynamic_state=run_impact(
        num_steps=STAGE6_DIAGNOSTIC_STEPS,save_stride=STAGE6_SAVE_EVERY,verbose=True)
    if dynamic_rows:
        with (STAGE6_DIR/'diagnostic_history.csv').open('w',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(dynamic_rows[0]))
            writer.writeheader();writer.writerows(dynamic_rows)
        max_force=max(r['impact_force_kN'] for r in dynamic_rows)
        max_defl=max(r['midspan_deflection_mm'] for r in dynamic_rows)
        print(f'DIAGNOSTIC ONLY: max stored contact force {max_force:.3f} kN')
        print(f'DIAGNOSTIC ONLY: max stored midspan deflection {max_defl:.3f} mm')
        if max(r['contact_points'] for r in dynamic_rows)==0:
            print('WARNING: no contact recorded; examine exact geometry and save frequency')
else:
    print('Diagnostic time integration not run. Set RUN_STAGE6_DIAGNOSTIC=True to run it.')


## E. Research interpretation and next fixes

**Priority 1 — source fidelity:** obtain/translate the Appendix C nonlinear interface force law and damage history; confirm interpolation point arrangement, rod location, effective interface area and nodal masses.

**Priority 2 — impact:** match Bowen's contact projection/reaction calculation and hemispherical hammer surface, including the exact initial condition and contact particle treatment; check support fixture and support reaction definitions.

**Priority 3 — rate-dependent concrete:** verify the strain-rate definition, `DIF_E` modification of modulus/micromodulus, both failure stretches and the update ordering against Chapter 2/Fortran. Current `bond_scalar_force` changes the thresholds using `DIF_t/DIF_E`, but does **not** independently multiply the elastic micromodulus by `DIF_E`; do not treat that as a confirmed exact implementation.

**Priority 4 — verification:** run elementary elastodynamics tests, convergence of force histories, discrete fracture-work accounting, unforced energy balance and a sensitivity study of the numerical contact stiffness.

**Priority 5 — validation:** only then compare with Bowen's Figure 3-3, Figure 3-4 and Table 3-1. Report the time histories, peak errors and structural fracture patterns, not merely two tuned peak values.


---
# Stage 7 — Bowen Appendix C interface, verified local translation

This stage adds the **central-force, trilinear-interpolated PDROD coupling** shown in Tu Bowen's Appendix C, pp. 89–91. Each interface interaction connects a steel node to a *concrete interpolation point*, not simply a steel node to a coincident concrete node.

**Fidelity status**: the force equation, interpolation, and irreversible `failfail` interpretation are source-derived. The number/placement of coupling sampling points, `bcc`, `s01`, `sc1`, integration weight `vol`, and exact force units are **not established by Appendix C alone**. We retain the Stage 4 sample layout as an *explicit provisional geometry* and specify an *independent* interface constitutive parameter object. The original appendix sets failure flags after force evaluation; this notebook enforces irreversibility before subsequent force evaluation (numerical repair, not a literal line-by-line copy).

The full dynamic run remains disabled until the missing parameters are resolved. This notebook is a **tested implementation step, not Bowen's reproduced impact curves**.

In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class AppendixCInterfaceParameters:
    bcc: float    # reference micromodulus (N/m^6 in a force-density interpretation)
    s01: float    # interface tensile-onset stretch
    sc1: float    # interface critical failure stretch
    horizon: float
    quadrature_volume: float   # Appendix C variable `vol`, NOT yet identified as exact interface quadrature

# Demonstration inputs ONLY, derived from concrete constants; do NOT claim Bowen's interface calibration.
DEMO_INTERFACE=AppendixCInterfaceParameters(
    bcc=float(72*p.E_concrete/(np.pi*(p.horizon_ratio*dx)**4)),
    s01=float(p.ft_concrete/p.E_concrete),
    sc1=float(5*p.Gf_concrete/(6*p.E_concrete*(p.horizon_ratio*dx)*(p.ft_concrete/p.E_concrete))),
    horizon=float(p.horizon_ratio*dx),
    quadrature_volume=float(V))
print('Demo interface parameters (NOT confirmed Bowen):', DEMO_INTERFACE)


In [ ]:
class AppendixCPDROD:
    """Source-shaped translation of the Appendix C nonlinear steel-to-concrete force.

    Sample is a virtual material point whose displacement is trilinearly interpolated
    from eight concrete points. Forces on the real concrete nodes are -N_a*F_steel.
    Persistent broken flags prevent crack healing on subsequent evaluations.
    """
    def __init__(self, steel_reference, virtual_reference, indices, weights, steel_ids, params):
        self.xs=np.asarray(steel_reference,dtype=float)
        self.xv=np.asarray(virtual_reference,dtype=float)
        self.indices=np.asarray(indices,dtype=np.int32)
        self.weights=np.asarray(weights,dtype=float)
        self.steel_ids=np.asarray(steel_ids,dtype=np.int32)
        self.params=params
        self.reference=self.xv-self.xs[self.steel_ids]
        self.lengths=np.linalg.norm(self.reference,axis=1)
        if self.indices.ndim!=2 or self.indices.shape[1]!=8: raise ValueError('eight interpolation weights required')
        if np.any(self.lengths<=0): raise ValueError('zero-length steel–concrete interface bond')
        if np.any(self.lengths>=params.horizon): raise ValueError('all coupling samples must lie inside horizon')
        np.testing.assert_allclose(self.weights.sum(axis=1),1.0,atol=1e-12)
        if not 0<params.s01<params.sc1: raise ValueError('invalid interface stretches')
        self.broken=np.zeros(len(self.lengths),dtype=bool)
        self.prev_stretch=np.zeros(len(self.lengths))

    def evaluate(self, concrete_u, steel_u, commit=False):
        concrete_u=np.asarray(concrete_u,dtype=float)
        steel_u=np.asarray(steel_u,dtype=float)
        interp=np.sum(concrete_u[self.indices]*self.weights[:,:,None],axis=1)
        deformed=self.reference+interp-steel_u[self.steel_ids]
        new_lengths=np.linalg.norm(deformed,axis=1)
        if np.any(new_lengths<=1e-16): raise FloatingPointError('collapsed interface bond')
        stretch=(new_lengths-self.lengths)/self.lengths
        prm=self.params
        c=prm.bcc*(1-(self.lengths/prm.horizon)**2)**2
        scalar=np.where(stretch<=prm.s01, c*stretch,
                    c*prm.s01*np.clip((prm.sc1-stretch)/(prm.sc1-prm.s01),0,1))
        newly_broken=stretch>prm.sc1
        effective_broken=self.broken | newly_broken
        scalar=np.where(effective_broken,0.,scalar)
        # Appendix C expresses the steel interaction as displacement toward the virtual point.
        sample_force=(scalar*prm.quadrature_volume)[:,None]*deformed/new_lengths[:,None]
        fsteel=np.zeros_like(steel_u)
        fconcrete=np.zeros_like(concrete_u)
        np.add.at(fsteel,self.steel_ids,sample_force)
        for a in range(8):
            np.add.at(fconcrete,self.indices[:,a],-self.weights[:,a,None]*sample_force)
        if commit:
            self.broken |= newly_broken
            self.prev_stretch[:]=stretch
        return fconcrete,fsteel,stretch,effective_broken

    def reset(self):
        self.broken.fill(False)
        self.prev_stretch.fill(0.)

pdrod_c=AppendixCPDROD(steel_xyz,sample_pos,sample_idx,sample_w,sample_steel,DEMO_INTERFACE)
print('Appendix C interactions:',len(pdrod_c.lengths),'initial broken:',pdrod_c.broken.sum())


In [ ]:
# Stage 7 independent verification of SOURCE-SHAPED interface law.
import numpy.testing as npt
rng7=np.random.default_rng(77)
uc0=np.zeros_like(xyz);us0=np.zeros_like(steel_xyz)
fc,fs,s,damage=pdrod_c.evaluate(uc0,us0)
assert np.max(np.abs(fc))==0 and np.max(np.abs(fs))==0
npt.assert_allclose(s,0,atol=1e-12)

trans=np.array([1e-5,2e-5,-3e-5])
fc,fs,_,_=pdrod_c.evaluate(uc0+trans,us0+trans)
assert np.linalg.norm(fc)+np.linalg.norm(fs)<1e-3

uc=rng7.normal(0,1e-8,xyz.shape); us=rng7.normal(0,1e-8,steel_xyz.shape)
fc,fs,_,_=pdrod_c.evaluate(uc,us)
net=fc.sum(axis=0)+fs.sum(axis=0)
scale=np.abs(fc).sum()+np.abs(fs).sum()
assert np.linalg.norm(net)<2e-12*max(scale,1.0)

# The forces are central to the interpolated location, so the total moment must also cancel.
mom=np.cross(xyz+uc,fc).sum(axis=0)+np.cross(steel_xyz+us,fs).sum(axis=0)
assert np.linalg.norm(mom)<1e-10*max(scale,1.0)

# Stretch beyond sc1: the bond breaks irreversibly; on unloading it cannot heal.
u_break=np.zeros_like(xyz);u_break[sample_idx[0]]=0.0
# Isolated one-sample reproduction removes coupling overlap while retaining the 8-point map.
unit=AppendixCPDROD(steel_xyz,sample_pos[:1],sample_idx[:1],sample_w[:1],sample_steel[:1],DEMO_INTERFACE)
u_steel=np.zeros_like(steel_xyz)
u_steel[unit.steel_ids[0]]=-2*DEMO_INTERFACE.sc1*unit.reference[0]
_,_,s_test,fail=unit.evaluate(uc0,u_steel,commit=True)
assert fail[0] and unit.broken[0]
_,fs_unload,_,fail_again=unit.evaluate(uc0,us0,commit=False)
assert fail_again[0] and np.linalg.norm(fs_unload)==0
print('PASS: zero state; translation; equal/opposite force; zero total moment; irreversible rupture')
print('PASS: Appendix C source-shaped nonlinear interface unit tests')


## Concrete rate-factor fidelity gate

The previously implemented function changes the failure thresholds using the rate factor, but does **not** separately multiply the elastic micromodulus by the modulus DIF. This helper isolates the constitutive modification so it can be independently tested. It does *not* silently change the existing dynamic solver; the proper constitutive selection and source-equivalent history update must still be confirmed.


In [ ]:
def bowen_concrete_force_with_modulus_DIF(stretch,stretch_rate,reference_distance,broken=False):
    """Thesis-informed correction candidate: apply DIF_E to elastic micromodulus AND DIF_t/DIF_E to stretches."""
    s,r,l,b=np.broadcast_arrays(np.asarray(stretch,float),np.asarray(stretch_rate,float),
                                 np.asarray(reference_distance,float),np.asarray(broken,bool))
    dif_t,dif_e=normal_concrete_difs(r)
    start=(p.ft_concrete/p.E_concrete)*dif_t/dif_e
    end=(5*p.Gf_concrete/(6*p.E_concrete*(p.horizon_ratio*p.dx)*(p.ft_concrete/p.E_concrete)))*dif_t/dif_e
    coeff=micromodulus(l)*dif_e
    strength=np.where(s<=start,coeff*s,coeff*start*np.clip((end-s)/(end-start),0,1))
    failed=b | (s>=end)
    return np.where(failed,0.,strength),failed

# Source-independent numeric consistency: elastic slope increases with DIF_E.
probe_s=1e-8
probe_l=0.5*p.horizon_ratio*p.dx
for rate in (0.,1e-3,1.):
    strength,_=bowen_concrete_force_with_modulus_DIF(probe_s,rate,probe_l)
    expected=micromodulus(probe_l)*normal_concrete_difs(rate)[1]*probe_s
    npt.assert_allclose(strength,expected,rtol=1e-12)
print('PASS: rate-dependent elastic-slope test')


## Integration choice and outstanding data

`AppendixCPDROD.evaluate()` is a runnable and tested nonlinear interface implementation, but its demonstration constants are **not** independently validated. For the full simulation, do not replace `coupling_forces` merely to obtain a closer peak force. First establish from the original Fortran implementation or author: (1) the actual `bcc, s01, sc1, vol, delta` and units for PDROD, (2) virtual sample spacing and orientations, (3) true reinforcement placement, (4) bond-history behavior under unloading, (5) impact contact and support details. When those are confirmed, inject this class using an explicit configuration switch and run numerical convergence tests before benchmarking.

**No Stage 7 test is a claim that the impact peak has been reproduced.**

---
# Stage 8 — Integrate source-shaped concrete and PDROD forces in impact dynamics

**Scope:** This stage **replaces** the Stage 5 linear interface and the earlier rate-independent elastic micromodulus in the time-stepping force path. It retains the discrete 33,600-point concrete model, steel rods, falling sphere and provisional supports/contact from Stage 5. The nonlinear interface constants/volume factor and impactor contact are still **not verified from Bowen's original Chapter 3 source**. Thus even a stable simulation is not an independent reproduction of published force or deflection.

**Important history-integrator decision:** Run the force law twice per Verlet step (before and after position update), but commit irreversible concrete and interface states **once** at the updated configuration. The strain-rate estimate uses the previous committed bond stretch. This avoids double-counting history updates and makes the algorithm testable.


In [ ]:
# Stage 8 uses the existing Numba conservative assembly and Stage 7 constitutive functions.
# Concrete forces are integrated over unique (i,j) bonds; PDROD returns actual nodal forces.
class SourceShapeConcrete(DiscretePDConcrete):
    def evaluate(self, displacement, dt, commit=False, override_rate=None):
        deformed=self.xyz+np.asarray(displacement,float)
        d=deformed[self.J]-deformed[self.I]
        cur=np.linalg.norm(d,axis=1)
        if np.any(cur<1e-15): raise FloatingPointError('Collapsed concrete bond')
        stretch=(cur-self.lengths)/self.lengths
        rate=(stretch-self.prev_stretch)/dt if override_rate is None else np.broadcast_to(override_rate,stretch.shape)
        scalar,failed=bowen_concrete_force_with_modulus_DIF(stretch,rate,self.lengths,self.broken)
        amplitude=scalar*(V**2)*self.weights
        f=accumulate_unique_pairs(self.I,self.J,d/cur[:,None],amplitude,len(self.xyz))
        if commit:
            self.broken|=failed
            self.prev_stretch[:]=stretch
        return f,stretch,failed

concrete8=SourceShapeConcrete(xyz,I,J,LEN,PAIR_WEIGHT)
interface8=AppendixCPDROD(steel_xyz,sample_pos,sample_idx,sample_w,sample_steel,DEMO_INTERFACE)

def reset_stage8():
    concrete8.broken.fill(False)
    concrete8.prev_stretch.fill(0.)
    interface8.reset()

def source_shaped_forces(uc,us,zh,dt,commit=False):
    pd_force, stretch, damaged=concrete8.evaluate(uc,dt,commit=commit)
    interface_c, interface_s, iface_stretch, iface_failed=interface8.evaluate(uc,us,commit=commit)
    rod_f=rod_forces(us)
    contact_c,contact_h,contact_U,ncontact,maxpen=sphere_contact(xyz+uc,zh)
    concrete_force=pd_force+interface_c+contact_c
    steel_force=rod_f+interface_s
    return concrete_force,steel_force,float(contact_h[2]),contact_U,ncontact,maxpen,int(damaged.sum()),int(iface_failed.sum())

print('Stage 8 integrated source-shaped forces READY')
print('WARNING: interface constants and contact law remain PROVISIONAL')

## A. Test coupled mechanics and irreversible history

These tests confirm that force transfer balances between the two material systems, that the interface is actually called from the integrated solver, and that failing bonds stay failed. They do not establish physical interface calibration.

In [ ]:
import numpy.testing as npt
reset_stage8()
zuc=np.zeros_like(xyz); zus=np.zeros_like(steel_xyz)
f0,s0,h0,E0,n0,pen0,nb0,ni0=source_shaped_forces(zuc,zus,hammer_z_initial,p.dt,commit=False)
assert np.max(np.abs(f0))==0 and np.max(np.abs(s0))==0
assert n0==0 and nb0==0 and ni0==0
# Rigid translation together of concrete/steel, without contact: no internal forces.
rig=np.array([2e-7,-1e-7,3e-7])
fr,sr,*_=source_shaped_forces(zuc+rig,zus+rig,hammer_z_initial+rig[2],p.dt,commit=False)
assert np.max(np.abs(fr))<1e-4 and np.max(np.abs(sr))<1e-4
# Apply a tiny relative steel movement; interface reactions must balance.
pert=zus.copy();pert[:,0]=1e-8
fc,fs,h,E,ns,mp,_,_=source_shaped_forces(zuc,pert,hammer_z_initial,p.dt,commit=False)
residual=np.linalg.norm(fc.sum(axis=0)+fs.sum(axis=0)+np.array([0.,0.,h]))
scale=max(np.linalg.norm(fc.sum(axis=0)),np.linalg.norm(fs.sum(axis=0)),1.)
assert residual/scale<1e-9, (residual,scale)
assert np.linalg.norm(fs)>0
assert np.count_nonzero(interface8.broken)==0  # read-only calls did not commit
print('PASS: zero-state and common rigid translation')
print('PASS: Stage 7 PDROD participates in integrated forces')
print(f'PASS: integrated pairwise action–reaction, relative residual={residual/scale:.3e}')
reset_stage8()

## B. Explicit integrator with a once-per-step history commit

The simulator has a strict upper bound and aborts on nonfinite fields, extreme displacement or excessive contact penetration. Reaction force is recorded at the hammer; dynamic energy is **not** asserted conserved here because neither the rate-dependent fracture work nor the dissipated interface energy is fully accounted for.

Set `RUN_STAGE8=True` to execute a short test. The **original 33,600-particle resolution** remains used even in short tests. Use the notebook on a sufficiently provisioned workstation for an extended run.

In [ ]:
def run_stage8(steps=5, stride=1, dt=p.dt, verbose=True):
    reset_stage8()
    uc=np.zeros_like(xyz);vc=np.zeros_like(xyz)
    us=np.zeros_like(steel_xyz);vs=np.zeros_like(steel_xyz)
    zh=float(hammer_z_initial);vh=float(HAMMER_V0)
    records=[]
    fc,fs,fh,contact_U,ncontact,pen,ncb,nib=source_shaped_forces(uc,us,zh,dt,commit=False)
    ac=fc/CONCRETE_NODE_MASS
    ast=fs/STEEL_MASS[:,None]
    ah=fh/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.)
    mid=(np.abs(xyz[:,0]-p.L/2)<=dx/2+1e-12)&(xyz[:,2]>=p.H/2)
    for step in range(1,steps+1):
        vc+=0.5*dt*ac;vs+=0.5*dt*ast;vh+=0.5*dt*ah
        uc+=dt*vc;us+=dt*vs;zh+=dt*vh
        apply_supports(uc,vc)
        fc,fs,fh,contact_U,ncontact,pen,ncb,nib=source_shaped_forces(uc,us,zh,dt,commit=True)
        ac=fc/CONCRETE_NODE_MASS;ast=fs/STEEL_MASS[:,None]
        ah=fh/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.)
        vc+=0.5*dt*ac;vs+=0.5*dt*ast;vh+=0.5*dt*ah
        apply_supports(uc,vc)
        if not (np.isfinite(uc).all() and np.isfinite(us).all() and np.isfinite(zh)):
            raise FloatingPointError(f'Nonfinite state at step {step}')
        maxu=np.max(np.abs(uc))
        if maxu>0.05:raise RuntimeError(f'Max concrete displacement {maxu:.3g} m at step {step}: aborting')
        if pen>0.5*dx:raise RuntimeError(f'Contact penetration {pen:.3g} m > 0.5 dx at step {step}')
        if step%stride==0 or step==1:
            rec=dict(step=step,time_ms=step*dt*1e3,
                     impact_force_kN=max(0.,fh)*1e-3,
                     midspan_deflection_mm=-np.mean(uc[mid,2])*1e3,
                     hammer_z_m=zh,hammer_velocity_mps=vh,
                     contact_count=ncontact,penetration_mm=pen*1e3,
                     concrete_broken=int(concrete8.broken.sum()),
                     interface_broken=int(interface8.broken.sum()),
                     kinetic_J=(0.5*CONCRETE_NODE_MASS*np.sum(vc*vc)
                         +0.5*np.sum(STEEL_MASS[:,None]*vs*vs)
                         +0.5*HAMMER_MASS*vh*vh),
                     contact_potential_J=contact_U)
            records.append(rec)
            if verbose:print(f"step={step:5d}, force={rec['impact_force_kN']:9.3f} kN, "
                             f"deflection={rec['midspan_deflection_mm']:.6f} mm, "
                             f"contacts={ncontact}, broken={rec['concrete_broken']}")
    return records,(uc,vc,us,vs,zh,vh)

RUN_STAGE8=False   # TURN ON only after reviewing the model limitations
STAGE8_STEPS=5     # 5-step test checks time stepping; it does not reach contact
STAGE8_STRIDE=1
if RUN_STAGE8:
    stage8_records,stage8_state=run_stage8(STAGE8_STEPS,STAGE8_STRIDE)
    from pathlib import Path
    import csv
    folder=Path('bowen_stage8_outputs');folder.mkdir(exist_ok=True)
    if stage8_records:
        with (folder/'stage8_diagnostics.csv').open('w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=stage8_records[0].keys());w.writeheader();w.writerows(stage8_records)
else:
    print('Stage 8 full dynamics NOT executed by default. Set RUN_STAGE8=True for a controlled test.')
    print('Estimated first-contact step from Stage 6:',first_step)


## C. Scientific readiness gate

**Now integrated:** 3-D concrete force assembly with rate-dependent micromodulus, Appendix C-shaped PDROD force transfer, irreversible failure flags, hammer dynamics and stagewise commit of damage states.

**Outstanding issues precluding a Bowen Chapter 3 reproduction claim:** (1) the original interface constants and interface quadrature multiplier are unknown; (2) truss/bar geometry and transverse reinforcement coordinates remain assumed; (3) contact penalty force is provisional rather than a demonstrated translation of his projection/reaction method; (4) boundary/support choices remain assumed; (5) tensile softening unloading history and rate-dependent fracture-energy budget are not fully formulated; (6) no end-to-end physical impact validation has yet been performed. Do not calibrate these unknowns exclusively to the two published peak values.

**Suggested next step:** obtain exact Appendix C code lines covering `bc`, `vol`, steel integration and the Chapter 3 contact/support steps, then replace the remaining provisional parameters before a full-duration benchmark.


---
# Stage 9 — Contact-phase readiness and reproducible diagnostic runs

**Scientific status:** This notebook uses the Stage 8 source-shaped PD concrete and PDROD equations, but the steel–concrete interface constants, hammer contact law, and some support choices remain provisional. The Stage 8 full-size 355-step trial exceeded a 200 s execution window before reaching first contact. **No peak force, deformation or fracture agreement with Bowen is claimed.**

**Workflow:** (1) confirm source-equation checks; (2) profile per-step runtime; (3) run the full-size stepper with an explicit time budget on your workstation; (4) export time histories; (5) check whether first contact occurs; (6) only then compare results against Chapter 3.

For first contact, the geometry-only estimate is approximately step 341. At this time deflection may still be negligible. Use many more steps after first contact for meaningful force and damage evolution. The initial hammer gap is a **provisional modelling choice** rather than confirmed source geometry.

In [ ]:
from time import perf_counter
from pathlib import Path
import csv, json

# Benchmark force evaluations independent of time integration.
# Note: repeated force evaluations at the undeformed state are not a stability test.
reset_stage8()
u0=np.zeros_like(xyz); s0=np.zeros_like(steel_xyz)
t0=perf_counter()
for _ in range(2):
    _=source_shaped_forces(u0,s0,float(hammer_z_initial),p.dt,commit=False)
force_eval_seconds=(perf_counter()-t0)/2
print(f'Full-mesh force evaluation: {force_eval_seconds:.3f} s')
print(f'Estimated 350-step minimum at two evaluations/step: {700*force_eval_seconds/60:.1f} min (ignores overhead)')
print('This timing depends on your CPU and is only a rough lower-bound estimate.')

## Optional run beyond first contact

The run below is **disabled by default**. Enable `RUN_STAGE9=True` on a capable local workstation. It retains the actual reference mesh; it does not use contact acceleration or artificially amplify the loading. The routine resets fracture history and begins from initial conditions every time it runs. Because the Stage 8 integrator does not expose restartable checkpoints, do **not** assume a CSV log is a restart state. Inspect the saved data after a completed run.

If the process exits with a maximum-displacement or penetration exception, **do not remove the safety guard**. Report the exception and the last record for further analysis.


In [ ]:
RUN_STAGE9 = False
STAGE9_STEPS = 450       # geometric contact ~341; just 109 steps beyond contact
STAGE9_STRIDE = 5

if RUN_STAGE9:
    started=perf_counter()
    records9, final_state9=run_stage8(STAGE9_STEPS, STAGE9_STRIDE, verbose=False)
    elapsed9=perf_counter()-started
    outdir=Path('bowen_stage9_outputs')
    outdir.mkdir(exist_ok=True)
    with (outdir/'impact_history.csv').open('w',newline='') as fp:
        writer=csv.DictWriter(fp,fieldnames=list(records9[0]))
        writer.writeheader();writer.writerows(records9)
    np.savez_compressed(outdir/'final_state.npz',
        uc=final_state9[0],vc=final_state9[1],us=final_state9[2],
        vs=final_state9[3],hammer_z=final_state9[4],hammer_v=final_state9[5],
        concrete_broken=concrete8.broken, interface_broken=interface8.broken,
        concrete_prev_stretch=concrete8.prev_stretch)
    r=np.array([v['impact_force_kN'] for v in records9]);
    d=np.array([v['midspan_deflection_mm'] for v in records9]);
    nc=np.array([v['contact_count'] for v in records9]);
    print('Wall time:',elapsed9,'s')
    print('First sampled contact step:',next((v['step'] for v in records9 if v['contact_count']>0),None))
    print('Maximum *sampled* impact force:',r.max(),'kN')
    print('Maximum *sampled* deflection magnitude:',np.abs(d).max(),'mm')
    print('Outputs:',outdir.resolve())
else:
    print('Stage 9 full-size contact run not executed. Set RUN_STAGE9=True to run.')

In [ ]:
if RUN_STAGE9:
    import matplotlib.pyplot as plt
    t=np.array([v['time_ms'] for v in records9]);
    fig,axs=plt.subplots(3,1,figsize=(9,9),sharex=True)
    axs[0].plot(t,[v['impact_force_kN'] for v in records9]);axs[0].set_ylabel('Contact force (kN)')
    axs[1].plot(t,[v['midspan_deflection_mm'] for v in records9]);axs[1].set_ylabel('Midspan displacement (mm)')
    axs[2].plot(t,[v['contact_count'] for v in records9]);axs[2].set_ylabel('Contacting particles');axs[2].set_xlabel('Time (ms)')
    fig.suptitle('Stage 9 — provisional contact model: diagnostic curves')
    fig.tight_layout();plt.show()

## Interpretation gate

- A `contact_count>0` and positive contact force would establish that the discrete contact activates; this alone does not establish accuracy.
- Compare the hammer's kinetic-energy reduction with growth of concrete/steel kinetic energy and dissipation. The present `kinetic_J` and `contact_potential_J` fields are insufficient for a closed energy audit; elastic bond energy, rod strain energy, irreversible fracture, and boundary work must be added.
- Do not tune penalty contact stiffness or unverified interface parameters to force peak agreement with **359.8 kN**. First recover Bowen's actual contact and interface parameters.
- Repeat with different mesh/horizon choices and check force-time and damage-pattern convergence after the source implementation is complete.


# Stage 10 — compiled source-shaped PD solver and contact-reaching test

This stage replaces **only the numerical implementation** of the Stage 8 concrete bond loop and Appendix C force distribution with Numba kernels. The nominal mesh, bond force law, rate DIFs, prescribed material values, steel elements and penalty contact are unchanged. **It is not claimed to be Bowen’s original Fortran or a verified full-impact result.**

The reference (NumPy) and compiled implementations are compared for the same displacement history before time integration. Persistent bond failure flags and stretch-history updates are checked separately. The optimized integrator retains the old provisional contact law, reinforcement sampling and boundary conditions. Any comparisons with Bowen's peak values remain exploratory until those source ambiguities are resolved.

In [ ]:
from numba import njit

@njit(cache=False)
def _fast_concrete_kernel(X, U, I0, J0, r0, pair_weight, prev_stretch, broken,
                          dt, E, ft, Gf, delta, fc, Vbond, commit):
    F = np.zeros((X.shape[0], 3), dtype=np.float64)
    failed_count = 0
    exp0 = 1.0/(10.0+6.0*fc/1.0e7)
    theta = 10.0**(6.0*exp0-2.0)
    reference_s0 = ft/E
    reference_sc = 5.0*Gf/(6.0*E*delta*reference_s0)
    base_c = 72.0*E/(np.pi*delta**4)
    for k in range(I0.size):
        i,j = I0[k],J0[k]
        ax = X[j,0]+U[j,0]-X[i,0]-U[i,0]
        ay = X[j,1]+U[j,1]-X[i,1]-U[i,1]
        az = X[j,2]+U[j,2]-X[i,2]-U[i,2]
        length = np.sqrt(ax*ax+ay*ay+az*az)
        if length <= 1.e-15:
            raise ValueError('collapsed concrete bond')
        stretch = (length-r0[k])/r0[k]
        rate = (stretch-prev_stretch[k])/dt
        if rate <= 3.e-6:
            dif_t=1.0
            dif_e=1.0
        else:
            ratio=rate/3.e-6
            dif_e=ratio**0.016
            if rate <= 1.0:
                dif_t=ratio**exp0
            else:
                dif_t=theta*ratio**(1.0/3.0)
        mult=dif_t/dif_e
        s0=reference_s0*mult
        sc=reference_sc*mult
        new_broken=broken[k] or stretch>=sc
        if new_broken:
            scalar=0.0
            failed_count += 1
        else:
            scaled=r0[k]/delta
            c=base_c*(1.0-scaled*scaled)**2*dif_e
            if stretch<=s0:
                scalar=c*stretch
            else:
                scalar=c*s0*(sc-stretch)/(sc-s0)
        amplitude=scalar*Vbond*pair_weight[k]/length
        fx=amplitude*ax; fy=amplitude*ay; fz=amplitude*az
        F[i,0]+=fx; F[i,1]+=fy; F[i,2]+=fz
        F[j,0]-=fx; F[j,1]-=fy; F[j,2]-=fz
        if commit:
            prev_stretch[k]=stretch
            if new_broken:
                broken[k]=True
    return F,failed_count

class FastConcrete:
    def __init__(self, other):
        self.broken=np.array(other.broken,copy=True)
        self.prev_stretch=np.array(other.prev_stretch,copy=True)
    def reset(self):
        self.broken[:]=False
        self.prev_stretch[:]=0.0
    def evaluate(self, U, timestep=p.dt, commit=False):
        return _fast_concrete_kernel(xyz,U,I,J,LEN,PAIR_WEIGHT,self.prev_stretch,self.broken,
                                     timestep,p.E_concrete,p.ft_concrete,p.Gf_concrete,
                                     delta,p.fc_concrete,V**2,commit)

fast_concrete=FastConcrete(concrete8)
print('Compiled PD concrete kernel defined for',len(I),'unique bonds')

In [ ]:
# Compare the SAME states in NumPy and compiled implementations, with no history commits.
from time import perf_counter
rng10=np.random.default_rng(110)
for label, amplitude in [('zero',0.0),('tiny',1.e-8),('elastic',1.e-7)]:
    U=rng10.normal(size=xyz.shape)*amplitude
    concrete8.broken.fill(False); concrete8.prev_stretch.fill(0.0)
    fast_concrete.reset()
    baseline=concrete8.evaluate(U,p.dt,commit=False)[0]
    optimized,nf=fast_concrete.evaluate(U,p.dt,commit=False)
    difference=np.max(np.abs(baseline-optimized))
    scale=max(np.max(np.abs(baseline)),1.)
    np.testing.assert_allclose(optimized,baseline,rtol=2.e-10,atol=1.e-7)
    print(f'{label}: max abs error={difference:.4g} N, normalized error={difference/scale:.3g}')

# Separate failure/history consistency test at high stretch, then unload.
U=np.zeros_like(xyz)
U[:,0]=5.e-3*(xyz[:,0]/p.L)**2
concrete8.broken.fill(False); concrete8.prev_stretch.fill(0.0)
fast_concrete.reset()
f_np,_,fail_np=concrete8.evaluate(U,p.dt,commit=True)
f_opt,nf=fast_concrete.evaluate(U,p.dt,commit=True)
assert np.array_equal(concrete8.broken,fast_concrete.broken)
np.testing.assert_allclose(f_np,f_opt,rtol=2.e-10,atol=1.e-7)
f_np2=concrete8.evaluate(np.zeros_like(U),p.dt,commit=True)[0]
f_opt2,nf2=fast_concrete.evaluate(np.zeros_like(U),p.dt,commit=True)
assert np.array_equal(concrete8.broken,fast_concrete.broken)
np.testing.assert_allclose(f_np2,f_opt2,rtol=2.e-10,atol=1.e-7)
print('PASS: irreversible history and unloading equivalence; broken bonds:',nf)
# Return original states to zeros for repeatable studies.
reset_stage8();fast_concrete.reset()

In [ ]:
@njit(cache=False)
def _fast_pdrod_kernel(uc,us,ref,ref_len,st_ids,indices,weights,broken,prev,
                       bcc,s01,sc1,horizon,quadrature_volume,commit):
    Fcon=np.zeros((uc.shape[0],3),dtype=np.float64)
    Fsteel=np.zeros((us.shape[0],3),dtype=np.float64)
    failed_count=0
    for k in range(ref_len.size):
        sid=st_ids[k]
        dxv=ref[k,0]-us[sid,0]
        dyv=ref[k,1]-us[sid,1]
        dzv=ref[k,2]-us[sid,2]
        for a in range(8):
            idx=indices[k,a]
            wt=weights[k,a]
            dxv+=wt*uc[idx,0]
            dyv+=wt*uc[idx,1]
            dzv+=wt*uc[idx,2]
        length=np.sqrt(dxv*dxv+dyv*dyv+dzv*dzv)
        if length<=1.e-15:
            raise ValueError('collapsed PDROD interface bond')
        stretch=(length-ref_len[k])/ref_len[k]
        failed=broken[k] or stretch>sc1
        if failed:
            scalar=0.0
            failed_count+=1
        else:
            c=bcc*(1.0-(ref_len[k]/horizon)**2)**2
            if stretch<=s01:
                scalar=c*stretch
            else:
                scalar=c*s01*(sc1-stretch)/(sc1-s01)
        amp=scalar*quadrature_volume/length
        fx=amp*dxv;fy=amp*dyv;fz=amp*dzv
        Fsteel[sid,0]+=fx;Fsteel[sid,1]+=fy;Fsteel[sid,2]+=fz
        for a in range(8):
            idx=indices[k,a];wt=weights[k,a]
            Fcon[idx,0]-=wt*fx;Fcon[idx,1]-=wt*fy;Fcon[idx,2]-=wt*fz
        if commit:
            broken[k]=failed
            prev[k]=stretch
    return Fcon,Fsteel,failed_count

class FastInterface:
    def __init__(self, original):
        self.original=original
        self.broken=np.zeros_like(original.broken)
        self.prev_stretch=np.zeros_like(original.prev_stretch)
    def reset(self):
        self.broken[:]=False;self.prev_stretch[:]=0.
    def evaluate(self,uc,us,commit=False):
        it=self.original;p0=it.params
        return _fast_pdrod_kernel(uc,us,it.reference,it.lengths,it.steel_ids,it.indices,
            it.weights,self.broken,self.prev_stretch,p0.bcc,p0.s01,p0.sc1,
            p0.horizon,p0.quadrature_volume,commit)

fast_interface=FastInterface(interface8)
rng10=np.random.default_rng(120)
uc=rng10.normal(size=xyz.shape)*1.e-8
us=rng10.normal(size=steel_xyz.shape)*1.e-8
interface8.reset();fast_interface.reset()
a,b,_,_=interface8.evaluate(uc,us,commit=False)
c,d,nfailed=fast_interface.evaluate(uc,us,commit=False)
np.testing.assert_allclose(a,c,rtol=1.e-5,atol=2.e-4)
np.testing.assert_allclose(b,d,rtol=1.e-5,atol=2.e-4)
print('PASS: fast eight-node nonlinear interface agrees with Stage 8')

In [ ]:
# Optimized solver: same loading and physical parameters, compiled force assembly.
# CONTACT LAW AND INTERFACE QUADRATURE REMAIN PROVISIONAL.
def fast_system_forces(uc,us,zh,dt,commit=False):
    Fc,nb=fast_concrete.evaluate(uc,dt,commit)
    Fic,Fis,ni=fast_interface.evaluate(uc,us,commit)
    Fs=rod_forces(us)+Fis
    Fc+=Fic
    C,H,Ucontact,ncontact,penetration=sphere_contact(xyz+uc,zh)
    Fc+=C
    return Fc,Fs,float(H[2]),Ucontact,ncontact,penetration,nb,ni

def run_fast_impact(steps=450,stride=5,dt=p.dt,verbose=True):
    fast_concrete.reset();fast_interface.reset()
    uc=np.zeros_like(xyz); vc=np.zeros_like(uc)
    us=np.zeros_like(steel_xyz);vs=np.zeros_like(us)
    zh=float(hammer_z_initial);vh=float(HAMMER_V0)
    FC,FS,FH,CU,NC,PEN,NB,NI=fast_system_forces(uc,us,zh,dt,False)
    ac=FC/CONCRETE_NODE_MASS;ast=FS/STEEL_MASS[:,None]
    ah=FH/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.)
    mid=(np.abs(xyz[:,0]-p.L/2)<=dx/2+1.e-12)&(xyz[:,2]>=p.H/2)
    recs=[];t0=perf_counter()
    for step in range(1,steps+1):
        vc+=0.5*dt*ac;vs+=0.5*dt*ast;vh+=0.5*dt*ah
        uc+=dt*vc;us+=dt*vs;zh+=dt*vh
        apply_supports(uc,vc)
        FC,FS,FH,CU,NC,PEN,NB,NI=fast_system_forces(uc,us,zh,dt,True)
        ac=FC/CONCRETE_NODE_MASS;ast=FS/STEEL_MASS[:,None]
        ah=FH/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.)
        vc+=0.5*dt*ac;vs+=0.5*dt*ast;vh+=0.5*dt*ah
        apply_supports(uc,vc)
        if not np.isfinite(uc).all() or not np.isfinite(us).all() or not np.isfinite(zh):
            raise FloatingPointError(f'Nonfinite state at step {step}')
        if np.max(np.abs(uc))>0.05 or PEN>0.5*dx:
            raise RuntimeError(f'Unphysical displacement/contact penetration at step {step}, pen={PEN:.5g} m')
        if step%stride==0 or step==1 or (NC and (not recs or recs[-1]['contact_count']==0)):
            q={'step':step,'time_ms':step*dt*1.e3,'force_kN':max(0.,FH)*1.e-3,
               'midspan_deflection_mm':-np.mean(uc[mid,2])*1.e3,'hammer_velocity_mps':vh,
               'contact_count':NC,'penetration_mm':PEN*1.e3,
               'concrete_broken':int(fast_concrete.broken.sum()),
               'interface_broken':int(fast_interface.broken.sum()),
               'kinetic_J':0.5*CONCRETE_NODE_MASS*np.sum(vc**2)+0.5*np.sum(STEEL_MASS[:,None]*vs**2)+0.5*HAMMER_MASS*vh**2,
               'contact_potential_J':CU}
            recs.append(q)
            if verbose and (step%50==0 or NC and len(recs)>1 and recs[-2]['contact_count']==0):
                print(f"step={step} t={q['time_ms']:.4f} ms contact={NC} F={q['force_kN']:.2f} kN defl={q['midspan_deflection_mm']:.4f} mm elapsed={perf_counter()-t0:.1f}s")
    return recs,(uc,vc,us,vs,zh,vh)

# Compilation warm-up, not counted in the solve runtime.
t0=perf_counter()
fast_concrete.evaluate(np.zeros_like(xyz),p.dt,False)
fast_interface.evaluate(np.zeros_like(xyz),np.zeros_like(steel_xyz),False)
print(f'Stage 10 compilation warm-up: {perf_counter()-t0:.2f}s')

# Explicit user-controlled run. Default is intentionally modest; full run uses 450.
RUN_STAGE10=False
STAGE10_STEPS=365  # 365-step first-contact interval verified; 450-step attempt failed at 371
STAGE10_STRIDE=5
if RUN_STAGE10:
    data10,state10=run_fast_impact(STAGE10_STEPS,STAGE10_STRIDE)
    import csv
    out10=Path('bowen_stage10_outputs');out10.mkdir(exist_ok=True)
    with open(out10/'impact_history.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=data10[0].keys());w.writeheader();w.writerows(data10)
    np.savez_compressed(out10/'final_state.npz', uc=state10[0],vc=state10[1],us=state10[2],vs=state10[3],
                        zh=state10[4],vh=state10[5],broken=fast_concrete.broken,interface_broken=fast_interface.broken)
    print('FIRST CONTACT:',next((x['step'] for x in data10 if x['contact_count']>0),None))
else:
    print('Set RUN_STAGE10=True to execute optimized contact-reaching simulation')

## Interpretation

The optimized solver is a **numerically equivalent implementation of the previous prototype on the tested states**, not independently validated against Bowen's original code. Do not report the simulated contact force as Bowen's predicted force until contact, coupling, support and fracture calibration are resolved. A first nonzero force, if obtained, establishes contact activation only. The complete impact duration is substantially longer than 450 time steps.

### Observed execution status (full reference mesh)

On the tested system, a controlled **365-step** run completed with first contact at step **341**, time **0.1023 ms**, four contact particles and approximately **0.46 kN** initial contact force. The maximum **sampled** force in that initial-contact interval was approximately **22.56 kN**. These numbers arise from a **provisional penalty-contact and reinforcement interface** and are **not** Bowen’s validated results. The 450-step attempt was deliberately stopped at **step 371** when the concrete displacement safety threshold of **50 mm** was exceeded; the reported contact penetration at that point was approximately **0.0275 mm**. The physical or numerical reason for the sudden displacement growth is **unresolved**. Do not simply raise the safety threshold and continue. Investigate strain-rate-based rupture, the damage law, support treatment, interface stiffness and stable time step first.

# Stage 11 — Dynamic instability localization and timestep audit

This notebook inherits Stages 1–10 and **does not claim to reproduce Bowen's impact result**. It diagnoses the first post-contact instability without raising the displacement safety threshold. Contact and steel–concrete interface inputs are still provisional.

Run `RUN_STAGE11 = True` to execute diagnostics. The default 390-step attempt stops at the original 50 mm criterion, saves the last stable state and a CSV with per-step diagnostics. A timestep comparison uses the same physical end time, not the same number of steps.

In [ ]:
from pathlib import Path
import csv

STAGE11_DIR = Path('bowen_stage11_outputs')
STAGE11_DIR.mkdir(exist_ok=True)

def diagnostic_run(dt=p.dt, t_end=0.000115, write_prefix='baseline', 
                   max_disp=0.05, max_pen=0.5*dx, verbose=True):
    fast_concrete.reset(); fast_interface.reset()
    uc=np.zeros_like(xyz); vc=np.zeros_like(uc)
    us=np.zeros_like(steel_xyz); vs=np.zeros_like(us)
    zh=float(hammer_z_initial); vh=float(HAMMER_V0)
    fc,fs,fh,cu,nc,pen,nb,ni=fast_system_forces(uc,us,zh,dt,False)
    ac=fc/CONCRETE_NODE_MASS; ast=fs/STEEL_MASS[:,None]
    ah=fh/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.)
    mid=(np.abs(xyz[:,0]-p.L/2)<=dx/2+1.e-12)&(xyz[:,2]>=p.H/2)
    records=[]; failure=None; snapshot=None
    for step in range(1,int(np.ceil(t_end/dt))+1):
        vc+=0.5*dt*ac; vs+=0.5*dt*ast; vh+=0.5*dt*ah
        uc+=dt*vc; us+=dt*vs; zh+=dt*vh
        apply_supports(uc,vc)
        fc,fs,fh,cu,nc,pen,nb,ni=fast_system_forces(uc,us,zh,dt,True)
        ac=fc/CONCRETE_NODE_MASS; ast=fs/STEEL_MASS[:,None]
        ah=fh/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.)
        vc+=0.5*dt*ac; vs+=0.5*dt*ast; vh+=0.5*dt*ah
        apply_supports(uc,vc)
        maxdisp=np.linalg.norm(uc,axis=1)
        maxidx=int(np.argmax(maxdisp)); maxacc=np.linalg.norm(ac,axis=1)
        maxai=int(np.argmax(maxacc));maxsi=int(np.argmax(np.linalg.norm(us,axis=1)))
        kinetic_con=0.5*CONCRETE_NODE_MASS*np.sum(vc*vc)
        kinetic_steel=0.5*np.sum(STEEL_MASS[:,None]*vs*vs)
        kinetic_ham=0.5*HAMMER_MASS*vh*vh
        q=dict(step=step,time_ms=step*dt*1e3,contact_kN=max(0,fh)*1e-3,
               contact_nodes=int(nc),penetration_mm=pen*1e3,
               midspan_mm=-float(np.mean(uc[mid,2]))*1e3,
               max_disp_mm=float(maxdisp[maxidx])*1e3,max_disp_node=maxidx,
               max_disp_x_mm=xyz[maxidx,0]*1e3,max_disp_y_mm=xyz[maxidx,1]*1e3,
               max_disp_z_mm=xyz[maxidx,2]*1e3,
               max_acc_mps2=float(maxacc[maxai]),max_acc_node=maxai,
               max_steel_disp_mm=float(np.linalg.norm(us[maxsi]))*1e3,
               max_steel_node=maxsi,concrete_broken=int(fast_concrete.broken.sum()),
               interface_broken=int(fast_interface.broken.sum()),
               concrete_kinetic_J=float(kinetic_con),steel_kinetic_J=float(kinetic_steel),
               hammer_kinetic_J=float(kinetic_ham),contact_potential_J=float(cu),
               hammer_velocity_mps=float(vh))
        records.append(q)
        finite=(np.isfinite(uc).all() and np.isfinite(us).all() and np.isfinite(zh))
        if not finite or maxdisp[maxidx]>max_disp or pen>max_pen:
            failure=f'At step {step}: max displacement {maxdisp[maxidx]*1e3:.3f} mm, penetration {pen*1e3:.5f} mm'
            break
        if verbose and (step%50==0 or (nc>0 and step>1 and records[-2]['contact_nodes']==0)):
            print(f'step={step} time={q["time_ms"]:.5f}ms F={q["contact_kN"]:.3f}kN maxU={q["max_disp_mm"]:.4f}mm maxA={q["max_acc_mps2"]:.3g}')
    with open(STAGE11_DIR/f'{write_prefix}_history.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(records[0]));w.writeheader();w.writerows(records)
    np.savez_compressed(STAGE11_DIR/f'{write_prefix}_last_state.npz',
        uc=uc,vc=vc,us=us,vs=vs,zh=zh,vh=vh,
        broken=fast_concrete.broken,interface_broken=fast_interface.broken)
    print('RUN:',write_prefix,'steps',len(records),'failure:',failure or 'none')
    return records,failure

RUN_STAGE11 = False
if RUN_STAGE11:
    stage11_baseline, baseline_failure=diagnostic_run(write_prefix='dt_3e-7')
    stage11_half, half_failure=diagnostic_run(dt=p.dt/2,
       write_prefix='dt_1p5e-7')
    import matplotlib.pyplot as plt
    fig,axs=plt.subplots(3,1,figsize=(9,10),sharex=True)
    for rr,label in [(stage11_baseline,'dt=3e-7 s'),(stage11_half,'dt=1.5e-7 s')]:
        tt=np.array([x['time_ms'] for x in rr]);
        axs[0].plot(tt,[x['contact_kN'] for x in rr],label=label)
        axs[1].plot(tt,[x['max_disp_mm'] for x in rr],label=label)
        axs[2].plot(tt,[x['max_acc_mps2'] for x in rr],label=label)
    for ax,lbl in zip(axs,['Contact force (kN)','Maximum concrete point displacement (mm)','Max concrete acceleration (m/s²)']):
        ax.set_ylabel(lbl);ax.grid(alpha=.3);ax.legend()
    axs[-1].set_xlabel('Physical time (ms)');fig.tight_layout()
    fig.savefig(STAGE11_DIR/'timestep_comparison.png',dpi=180)
    print('Outputs saved to',STAGE11_DIR)
else:
    print('Stage 11 ready. Set RUN_STAGE11=True for both diagnostic runs.')


# Stage 12 — Isolating the steel–concrete coupling instability

**Research goal.** Run identical full-resolution early-impact simulations with (A) concrete only, (B) steel rods linked to concrete by an explicitly provisional weak elastic connection, and (C) the Stage 11 nonlinear PDROD coupling. The only intended differences are the interface terms. Stage 12 is a *numerical fault-isolation experiment*, **not** a validated reproduction of Bowen's Chapter 3 response.

**Important limits:** Contact stiffness, reinforcement geometry, and PDROD bond quadrature are provisional. The concrete-only case neglects reinforcement deliberately. The elastic-link case uses 1e5 N/m per coupling sample purely as a diagnostic, not as a calibrated constitutive property. All cases share identical hammer, particle geometry, supports, concrete force kernel and time step. Full-record stopping limits are used to avoid describing a blown-up solution as an impact result.


# Stage 14 — Full-beam adapter for Bowen Appendix C

**Purpose**: swap the *printed* Appendix C steel–concrete force law into the Stage 10/12 beam dynamical solver, preserving the source's force-before-failure-flag update, and compare with the previously modified irreversible coupling. This does not supply the missing original Appendix C `bcc`, `s01`, `sc1`, `vol`, reference connectivity, or support/contact definitions. **No exact-result claim is possible without those original data.**

Stage 12's earlier auto-running comparison cells have deliberately been excluded. All new long simulations default to OFF. The beam discretization and demonstration interface data are still those in the preceding cells.


In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass(frozen=True)
class AppendixCParameters:
    bcc: float
    s01: float
    sc1: float
    delta: float
    vol: float

class AppendixCPublishedKernel:
    """Vectorized form of Appendix C using explicit 0-based steel/concrete indexing."""
    def __init__(self, steel_xyz, concrete_xyz, steel_id, concrete8_id,
                 weights8, virtual_xyz, params: AppendixCParameters):
        self.xs = np.asarray(steel_xyz, float)
        self.xc = np.asarray(concrete_xyz, float)
        self.st = np.asarray(steel_id, np.int64)
        self.idx = np.asarray(concrete8_id, np.int64)
        self.w = np.asarray(weights8, float)
        self.xv = np.asarray(virtual_xyz, float)
        self.p = params
        if self.idx.shape != (len(self.st), 8) or self.w.shape != self.idx.shape:
            raise ValueError('Each coupling entry must have exactly 8 concrete indices and weights')
        if self.xv.shape != (len(self.st), 3): raise ValueError('virtual_xyz must have shape (K,3)')
        if np.any(self.st<0) or np.any(self.st>=len(self.xs)) or np.any(self.idx<0) or np.any(self.idx>=len(self.xc)):
            raise IndexError('Input connectivity is outside the respective steel or concrete array')
        if np.max(np.abs(self.w.sum(axis=1)-1))>1e-10: raise ValueError('Shape weights must sum to one')
        self.reference = self.xv-self.xs[self.st]
        self.idist = np.linalg.norm(self.reference, axis=1)
        if np.any(self.idist<=0): raise ValueError('PDROD bonds cannot have zero initial length')
        if not (params.bcc>0 and params.delta>0 and params.vol>0 and 0<params.s01<params.sc1):
            raise ValueError('Invalid PDROD constants')
        if np.any(self.idist>params.delta): raise ValueError('Neighbors must be within delta')
        self.failfail = np.ones(len(self.st), dtype=np.int8)

    def evaluate(self, uc, us, old_uc=None, old_us=None, dt=None,
                 mode='literal', commit=False):
        if mode not in ('literal','irreversible_corrected'):
            raise ValueError('mode must be literal or irreversible_corrected')
        uc=np.asarray(uc,float); us=np.asarray(us,float)
        if uc.shape != self.xc.shape or us.shape != self.xs.shape:
            raise ValueError('Displacements must match geometry shape')
        dispdisp=np.einsum('ka,kad->kd',self.w, uc[self.idx])
        direction=self.reference+dispdisp-us[self.st]
        nlength=np.linalg.norm(direction,axis=1)
        if np.any(nlength<1e-15): raise FloatingPointError('Current interface length is zero')
        sssss=(nlength-self.idist)/self.idist
        if old_uc is not None and old_us is not None and dt is not None:
            if dt<=0: raise ValueError('dt must be positive')
            old_interp=np.einsum('ka,kad->kd',self.w,np.asarray(old_uc)[self.idx])
            oldnlength=np.linalg.norm(self.reference+old_interp-np.asarray(old_us)[self.st],axis=1)
            s1=np.abs((nlength-oldnlength)/self.idist/dt)
        else:
            oldnlength=np.full_like(nlength,np.nan); s1=np.full_like(nlength,np.nan)
        bc=self.p.bcc*(1-(self.idist/self.p.delta)**2)**2
        scalar=np.where(sssss<=self.p.s01,bc*sssss,
                    np.where(sssss<=self.p.sc1,
                        bc*self.p.s01*(self.p.sc1-sssss)/(self.p.sc1-self.p.s01),0.0))
        new_failed=sssss>self.p.sc1
        if mode=='irreversible_corrected':
            scalar=np.where((self.failfail==0)|new_failed,0.,scalar)
        # Appendix C: dforce = bc*(stretch law)*vol*(deformed vector)/nlength
        dforce=(scalar*self.p.vol)[:,None]*(direction/nlength[:,None])
        fs=np.zeros_like(us); fc=np.zeros_like(uc)
        np.add.at(fs,self.st,dforce)
        for a in range(8): np.add.at(fc,self.idx[:,a],-self.w[:,a,None]*dforce)
        # Printed code computes failure flag after dforce and does not use it above.
        if commit: self.failfail[new_failed]=0
        dmgpar1=float(np.sum(self.failfail*self.p.vol))
        dmgpar2=float(len(self.failfail)*self.p.vol)
        return dict(concrete_force=fc,steel_force=fs,stretch=sssss,stretch_rate=s1,
                    oldnlength=oldnlength,current_length=nlength,dforce=dforce,
                    new_failed=new_failed,failfail=self.failfail.copy(),
                    sample_damage=(1-dmgpar1/dmgpar2 if dmgpar2 else np.nan))
    def reset(self): self.failfail[:]=1


In [ ]:
# Bridge the source-shaped Stage 13 class into Stage 12/10 beam arrays.
# IMPORTANT: sample_pos, sample_idx, sample_w and DEMO_INTERFACE are PROVISIONAL
# from earlier setup. Appendix C does not give their original Chapter 3 values.
SOURCE_C_PARAMS=AppendixCParameters(
    bcc=DEMO_INTERFACE.bcc,
    s01=DEMO_INTERFACE.s01,
    sc1=DEMO_INTERFACE.sc1,
    delta=DEMO_INTERFACE.horizon,
    vol=DEMO_INTERFACE.quadrature_volume,
)
appendix_c_beam=AppendixCPublishedKernel(
    steel_xyz,xyz,sample_steel,sample_idx,sample_w,sample_pos,SOURCE_C_PARAMS)
assert len(appendix_c_beam.st)==len(sample_steel)
print('Stage 14 Appendix C beam samples:',len(sample_steel),' UNVERIFIED constants:',SOURCE_C_PARAMS)


In [ ]:
# Direct comparison: source-literal interface vs earlier irreversible interface
# in elastic range, both should agree numerically (same demo inputs and mesh).
uf=np.zeros_like(xyz); sf=np.zeros_like(steel_xyz)
r0=appendix_c_beam.evaluate(uf,sf,mode='literal')
assert np.max(np.abs(r0['concrete_force']))==0.
assert np.max(np.abs(r0['steel_force']))==0.

rng14=np.random.default_rng(147)
uf=rng14.normal(0,1.e-8,xyz.shape);sf=rng14.normal(0,1.e-8,steel_xyz.shape)
r1=appendix_c_beam.evaluate(uf,sf,mode='literal')
fast_interface.reset()
fc_old,fs_old,_=fast_interface.evaluate(uf,sf,False)
np.testing.assert_allclose(r1['concrete_force'],fc_old,rtol=1e-8,atol=2e-4)
np.testing.assert_allclose(r1['steel_force'],fs_old,rtol=1e-8,atol=2e-4)
assert np.linalg.norm(r1['concrete_force'].sum(axis=0)+r1['steel_force'].sum(axis=0))<1e-4
print('PASS zero state, source-vs-previous elastic force agreement, and total force equilibrium')


In [ ]:
# Explicitly demonstrate the known key behavioral difference after a bond breaks.
appendix_c_beam.reset();fast_interface.reset()
# Deform only the steel node associated with one sample, temporarily;
# many interactions on this node may fail, which is expected.
uf=np.zeros_like(xyz);sf=np.zeros_like(steel_xyz)
first=int(sample_steel[0]);sf[first,0]=1.0e-3
appendix_c_beam.evaluate(uf,sf,mode='literal',commit=True)
fast_interface.evaluate(uf,sf,commit=True)
assert np.count_nonzero(appendix_c_beam.failfail==0)>0
uf.fill(0.);sf.fill(0.);sf[first,0]=1.0e-7
literal_after=appendix_c_beam.evaluate(uf,sf,mode='literal')['steel_force'][first].copy()
corrected_after=appendix_c_beam.evaluate(uf,sf,mode='irreversible_corrected')['steel_force'][first].copy()
fc_prev,fs_prev,_=fast_interface.evaluate(uf,sf,False)
print('Broken interface samples:',np.count_nonzero(appendix_c_beam.failfail==0))
print('Re-loading nodal steel force, literal (N):',literal_after)
print('Re-loading nodal steel force, history-corrected (N):',corrected_after)
print('Previously accelerated irreversible code (N):',fs_prev[first])
appendix_c_beam.reset();fast_interface.reset()


## Impact integration using Appendix C directly

The function below uses the existing Stage 10 concrete kernel, rod forces, impactor contact and support functions. Only the interface force evaluation is replaced. It returns forces and histories with a safety stop. **The `literal` mode follows the Appendix C excerpt's failure-flag handling; `irreversible_corrected` is a transparent nonliteral modification.**

The interface force `vol` scaling remains unverified and may cause instability. Accordingly the impact run is OFF unless explicitly activated. Testing on the benchmark is a diagnostic experiment, not validation.


In [ ]:
from time import perf_counter
import csv
from pathlib import Path
OUT14=Path('bowen_stage14_outputs');OUT14.mkdir(exist_ok=True)

def source_c_force(uc,us,mode,commit=False):
    result=appendix_c_beam.evaluate(uc,us,mode=mode,commit=commit)
    return result['concrete_force'], result['steel_force'],int((result['failfail']==0).sum())

def run_stage14(mode='literal',steps=380,dt=p.dt,stride=5,limit_mm=50.):
    if mode not in ('literal','irreversible_corrected'):raise ValueError(mode)
    appendix_c_beam.reset(); fast_concrete.reset()
    uc=np.zeros_like(xyz);vc=np.zeros_like(uc)
    us=np.zeros_like(steel_xyz);vs=np.zeros_like(us)
    zh=float(hammer_z_initial);vh=float(HAMMER_V0)
    mid=(np.abs(xyz[:,0]-p.L/2)<=dx/2+1.e-12)&(xyz[:,2]>=p.H/2)
    def forces(commit):
        fc,n_b=fast_concrete.evaluate(uc,dt,commit)
        fi,fs,n_i=source_c_force(uc,us,mode,commit)
        fc+=fi
        fs+=rod_forces(us)
        ct,fh,cu,nc,pen=sphere_contact(xyz+uc,zh)
        fc+=ct
        return fc,fs,float(fh[2]),int(nc),float(pen),int(n_b),int(n_i)
    fc,fs,fh,nc,pen,nb,ni=forces(False)
    ac=fc/CONCRETE_NODE_MASS;ast=fs/STEEL_MASS[:,None]
    ah=fh/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.)
    rows=[];first_contact=None;stop=None;t0=perf_counter()
    for step in range(1,steps+1):
        vc+=0.5*dt*ac;vs+=0.5*dt*ast;vh+=0.5*dt*ah
        uc+=dt*vc;us+=dt*vs;zh+=dt*vh
        apply_supports(uc,vc)
        fc,fs,fh,nc,pen,nb,ni=forces(True)
        ac=fc/CONCRETE_NODE_MASS;ast=fs/STEEL_MASS[:,None]
        ah=fh/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.)
        vc+=0.5*dt*ac;vs+=0.5*dt*ast;vh+=0.5*dt*ah
        apply_supports(uc,vc)
        if nc and first_contact is None:first_contact=step
        max_c=float(np.max(np.linalg.norm(uc,axis=1)))*1e3
        max_s=float(np.max(np.linalg.norm(us,axis=1)))*1e3
        if (not np.isfinite(uc).all()) or (not np.isfinite(us).all()) or max(max_c,max_s)>limit_mm or pen>0.5*dx:
            stop=f'safety stop at step {step}: max_c={max_c:.4g} mm max_s={max_s:.4g} mm'
        if step==1 or step%stride==0 or nc and step==first_contact or stop:
            rows.append(dict(step=step,time_ms=step*dt*1000,force_kN=max(0.,fh)*1e-3,
                contact_points=nc,midspan_mm=-1000*np.mean(uc[mid,2]),
                max_concrete_mm=max_c,max_steel_mm=max_s,broken_concrete=nb,
                broken_interface=ni,hammer_v_mps=vh,penetration_mm=pen*1000))
        if stop:break
    output=OUT14/f'{mode}_history.csv'
    with output.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    summary=dict(mode=mode,steps_executed=step,first_contact_step=first_contact,
                 elapsed_s=perf_counter()-t0,stop=stop,
                 peak_recorded_kN=max(x['force_kN'] for x in rows))
    print('Stage 14:',summary)
    return rows,summary

RUN_STAGE14=False
if RUN_STAGE14:
    # Run separately: each call resets concrete and interface histories.
    result_literal,summary_literal=run_stage14('literal',steps=380)
    result_corrected,summary_corrected=run_stage14('irreversible_corrected',steps=380)
else:
    print('Stage 14 dynamic run is OFF. Set RUN_STAGE14=True to test the unverified benchmark inputs.')


### Acceptance criteria before numerical reproduction

- Obtain Bowen's original values for `bcc`, `s01`, `sc1`, `vol`, steel–concrete neighbor/weight connectivity, and actual steel node positions.
- Establish whether the full original Fortran code checks `failfail` elsewhere (the Appendix C excerpt alone does not).
- Confirm original hammer and supports; our current implementation remains provisional.
- Verify force dimensions and interface tangent stiffness against steel nodal inertia.
- Compare snapshots, crack patterns, force history, and midspan deflection to thesis after the preceding checks.

**No claim is made that this adapter alone reproduces the original results.**

# Stage 17 — Corrected interface acceleration normalization (diagnostic)

**Source convention:** The supplied `Console2.f90` divides its PD interface force-density contributions by the material density. In prior Python notebooks those same interface contributions were treated as physical nodal forces and divided by steel lumped mass, a major scale mismatch. 

**This notebook keeps the existing concrete PD and rod axial forces in physical nodal-force form**. It treats **only** the Appendix C interaction vector as a *force-density entry* and divides those entries by `rho_concrete` or `rho_steel`, while the remaining physical nodal forces divide by their appropriate nodal masses. This is a hybrid translation designed for source-convention diagnosis, **not** a proven physical discretization (source array/volume units and total momentum must still be checked). The contact and coupling parameters remain provisional.

Do not interpret stable curves as a validation of Bowen's original Chapter 3 benchmark.

In [ ]:
from pathlib import Path
import csv, json, time
import numpy as np
STAGE17_DIR=Path('bowen_stage17_outputs')
STAGE17_DIR.mkdir(exist_ok=True)

def normalized_acceleration(physical_concrete, physical_steel, interface_concrete_density,
                            interface_steel_density):
    """Only the Appendix C entries are force density; all other inputs are nodal N."""
    return (physical_concrete/CONCRETE_NODE_MASS + interface_concrete_density/p.rho_concrete,
            physical_steel/STEEL_MASS[:,None] + interface_steel_density/p.rho_steel)

# Check the normalization independently of contact and fracture.
fq=np.array([[74204.,0.,0.]])
a_source=fq/p.rho_steel
m=STEEL_MASS[0]
a_wrong=fq/m
assert np.linalg.norm(a_wrong)>np.linalg.norm(a_source)
np.testing.assert_allclose(a_source, fq*STEEL_MASS[0]/p.rho_steel/STEEL_MASS[0])
print('PASS: source acceleration = force-density / steel density')
print('Example source acceleration:',a_source[0,0], 'm/s^2, previous lumped mass:',a_wrong[0,0])

In [ ]:
def stage17_components(uc,us,zh,dt,commit=False,mode='irreversible_corrected'):
    """Separate physical nodal force contributions from source-shaped interface density entries."""
    fcp,broken_con=fast_concrete.evaluate(uc,dt,commit)
    fsp=rod_forces(us)
    fi_c,fi_s,ni=source_c_force(uc,us,mode,commit)
    contact_c,hammer_f,contact_U,ncontact,maxpen=sphere_contact(xyz+uc,zh)
    fcp+=contact_c
    return fcp,fsp,fi_c,fi_s,float(hammer_f[2]),int(ncontact),float(maxpen),int(broken_con),int(ni)

def stage17_run(steps=420, dt=p.dt, save_every=2,mode='irreversible_corrected',limit_mm=50.,verbose=True):
    assert mode in ('literal','irreversible_corrected')
    appendix_c_beam.reset();fast_concrete.reset()
    uc=np.zeros_like(xyz);vc=np.zeros_like(xyz)
    us=np.zeros_like(steel_xyz);vs=np.zeros_like(steel_xyz)
    zh=float(hammer_z_initial);vh=float(HAMMER_V0)
    mid=(np.abs(xyz[:,0]-p.L/2)<=dx/2+1e-12)&(xyz[:,2]>=p.H/2)
    def force_step(commit):
        physical_c,physical_s,density_c,density_s,fh,nc,pen,nb,ni=stage17_components(uc,us,zh,dt,commit,mode)
        ac,ast=normalized_acceleration(physical_c,physical_s,density_c,density_s)
        ah=fh/HAMMER_MASS-(9.81 if USE_GRAVITY else 0.0)
        physical_interface_sum=density_c.sum(axis=0)*dx**3 + np.sum(density_s*STEEL_MASS[:,None]/p.rho_steel,axis=0)
        return ac,ast,ah,fh,nc,pen,nb,ni,float(np.linalg.norm(physical_interface_sum)),float(np.max(np.linalg.norm(ast,axis=1)))
    ac,ast,ah,fh,nc,pen,nb,ni,mom_error,amax_s=force_step(False)
    rows=[];error=None;first_contact=None;t0=time.perf_counter()
    for step in range(1,steps+1):
        vc+=.5*dt*ac;vs+=.5*dt*ast;vh+=.5*dt*ah
        uc+=dt*vc;us+=dt*vs;zh+=dt*vh
        apply_supports(uc,vc)
        ac,ast,ah,fh,nc,pen,nb,ni,mom_error,amax_s=force_step(True)
        vc+=.5*dt*ac;vs+=.5*dt*ast;vh+=.5*dt*ah
        apply_supports(uc,vc)
        if nc and first_contact is None:first_contact=step
        mc=float(np.max(np.linalg.norm(uc,axis=1)))*1000
        ms=float(np.max(np.linalg.norm(us,axis=1)))*1000
        if not all(np.isfinite(z) for z in (mc,ms,fh,amax_s)) or max(mc,ms)>limit_mm or pen>dx/2:
            error=f'safety stop step {step}: concrete={mc:.4g} mm; steel={ms:.4g} mm; penetration={pen*1000:.4g} mm'
        if step==1 or step%save_every==0 or error or (first_contact==step):
            rows.append(dict(step=step,time_ms=step*dt*1000,impact_kN=max(0,fh)/1000,
                 midspan_mm=-1000*np.mean(uc[mid,2]),max_concrete_mm=mc,max_steel_mm=ms,
                 n_contact=nc,penetration_mm=pen*1000,broken_concrete=nb,
                 broken_interface=ni,max_steel_acceleration=amax_s,
                 interface_physical_momentum_residual_N=mom_error,hammer_velocity=vh))
        if verbose and (step%50==0 or error): print(step,'force kN',round(fh/1000,3),'max concrete mm',round(mc,6),'max steel mm',round(ms,6),'interface failures',ni)
        if error:break
    summary=dict(mode=mode,steps=int(step),first_contact=first_contact,stop=error,
                 max_recorded_contact_kN=max(r['impact_kN'] for r in rows),
                 max_concrete_mm=max(r['max_concrete_mm'] for r in rows),
                 max_steel_mm=max(r['max_steel_mm'] for r in rows),elapsed_s=time.perf_counter()-t0,
                 source_status='source-density convention adapted; interface, contact and geometry provisional')
    with (STAGE17_DIR/'density_normalized_history.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    (STAGE17_DIR/'summary.json').write_text(json.dumps(summary,indent=2))
    np.savez_compressed(STAGE17_DIR/'state.npz',uc=uc,vc=vc,us=us,vs=vs,hammer_z=zh,hammer_velocity=vh)
    print('Stage17 summary:',summary)
    return rows,summary

# Conservation warning: physical interface momentum is not guaranteed by force-density parity
r=np.random.default_rng(17)
u0=r.normal(0,1.e-8,size=xyz.shape);s0=r.normal(0,1.e-8,size=steel_xyz.shape)
fc_d,fs_d,_=source_c_force(u0,s0,'literal',commit=False)
print('Interface force-density vector sum, for regression only:', np.linalg.norm(fc_d.sum(axis=0)+fs_d.sum(axis=0)))
print('Physical force residual under mixed control volumes:',np.linalg.norm(fc_d.sum(axis=0)*dx**3+np.sum(fs_d*STEEL_MASS[:,None]/p.rho_steel,axis=0)))

In [ ]:
RUN_STAGE17 = False  # Set True for contact-reaching diagnostic.
STAGE17_STEPS = 420
if RUN_STAGE17:
    history17,summary17=stage17_run(steps=STAGE17_STEPS,save_every=2)
else:
    print('Stage 17 is ready. Set RUN_STAGE17=True to run impact diagnostics.')

## Interpretation and limitations

This density-based coupling path addresses the **specific** discrepancy between a Fortran force-density update and the previous Python lumped-mass update. It does **not** establish a consistent variational interface coupling between particles of different effective volumes. The physical-momentum residual is explicitly measured and must be checked. Never combine the stable result with published reference curves until interface coefficients, interaction volumes, contact algorithm, and steel nodal layout are recovered from the actual Chapter 3 code.